# Code below divided into 4 sections:
#### Fetcher, Classifier, Storage & Agent


Install any missing libraries

In [ ]:
!pip -q install \
  pymupdf \
  groq \
  python-dotenv \
  httpx \
  feedparser \
  pandas \
  openpyxl \
  tqdm \
  gspread \
  google-auth \
  google-auth-oauthlib \
  google-auth-httplib2 \
  google-api-python-client

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.1 MB/s eta 0:00:00


### To run this, we need a groq key
### Step by step guide to get a free access key:
1.   Enter this website https://console.groq.com/home
2.   Create an account (avoid hotmail, outlook or live because it won't work)
3. go to API Keys > create API Key > Save the key somewhere you have access to

In [ ]:
# Enter Groq API key
import os
from getpass import getpass

# Hidden input — nothing is shown while typing
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("Key loaded successfully ✅")

Enter your Groq API key: ··········
Key loaded successfully ✅


#1. Fetcher

In [ ]:
import os
import csv
import time
import random
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Sequence

import httpx
import feedparser

ARXIV_API = "https://export.arxiv.org/api/query"

DEFAULT_CATEGORIES = [
    # Core ML / AI
    "cs.AI",
    "cs.LG",
    "stat.ML",
    # Medical imaging
    "cs.CV",
    "eess.IV",
    # Biosignals
    "eess.SP",
    # Biomedical / genomics
    "q-bio.GN",
    "q-bio.NC",
    "q-bio.BM",
    "q-bio.TO",
    # Medical physics
    "physics.med-ph",
    "physics.bio-ph",
    # Epidemiology / stats
    "math.ST",
]

# -------------------------
# Helpers: dates + queries
# -------------------------
def _to_arxiv_dt(dt: datetime) -> str:
    """arXiv date format: YYYYMMDDHHMM"""
    return dt.strftime("%Y%m%d%H%M")


def _parse_yyyy_mm_dd(s: str) -> datetime:
    """Parse 'YYYY-MM-DD' into datetime at midnight."""
    return datetime.strptime(s, "%Y-%m-%d")


def _build_query(categories: Sequence[str], date_from: Optional[str], date_to: Optional[str]) -> str:
    cat_part = "(" + " OR ".join([f"cat:{c}" for c in categories]) + ")"
    if date_from and date_to:
        return f"{cat_part} AND submittedDate:[{date_from} TO {date_to}]"
    return cat_part


# Entry normalization
# -------------------------
def _parse_entry(entry) -> Dict:
    entry_id = entry.get("id", "") or ""
    arxiv_id = entry_id.split("/abs/")[-1] if "/abs/" in entry_id else entry_id.split("/")[-1]

    pdf_url = ""
    for link in entry.get("links", []) or []:
        href = getattr(link, "href", "") or ""
        ltype = getattr(link, "type", "") or ""
        if ltype == "application/pdf":
            pdf_url = href
            break

    categories = []
    for t in (entry.get("tags") or []):
        term = None
        if isinstance(t, dict):
            term = t.get("term")
        else:
            term = getattr(t, "term", None)
        if term:
            categories.append(term)

    return {
        "arxiv_id": arxiv_id,
        "id": entry_id,
        "title": (entry.get("title", "") or "").strip().replace("\n", " "),
        "abstract": (entry.get("summary", "") or "").strip().replace("\n", " "),
        "authors": [a.name for a in entry.get("authors", [])] if entry.get("authors") else [],
        "published": entry.get("published", "") or "",
        "updated": entry.get("updated", "") or "",
        "link": entry.get("link", "") or entry_id,
        "pdf_url": pdf_url,
        "categories": categories,
    }

# Robust HTTP fetch w/ retry
# -------------------------
def _get_with_retry(
    client: httpx.Client,
    params: Dict,
    max_attempts: int = 10,
    base_sleep: float = 15.0,
) -> str:
    last_err = None

    for attempt in range(1, max_attempts + 1):
        try:
            r = client.get(ARXIV_API, params=params)
            r.raise_for_status()
            return r.text

        except httpx.HTTPStatusError as e:
            last_err = e
            status = e.response.status_code

            # If arXiv rate-limits us, back off harder
            if status == 429:
                sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5)
                print(f"[429] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
            else:
                sleep_s = base_sleep + random.uniform(0.25, 0.75)
                print(f"[HTTP {status}] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)

        except (httpx.TimeoutException, httpx.TransportError) as e:
            last_err = e
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5)
            print(f"[Network retry] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)

        except Exception as e:
            last_err = e
            sleep_s = base_sleep * (2 ** (attempt - 1)) + random.uniform(0.5, 1.5)
            print(f"[Retry] attempt {attempt}/{max_attempts} -> sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {max_attempts} attempts. Last error: {last_err}")


# Main fetcher
# -------------------------
def fetch_papers_by_date_range(
    start_date: str,
    end_date: str,
    categories: Optional[List[str]] = None,
    max_results_total: int = 500,
    page_size: int = 25,
    chunk_days: int = 31,
    timeout: int = 60,
    polite_delay_seconds: float = 15.0,
    start_offset: int = 0,
) -> List[Dict]:
    """
    Fetch arXiv papers for chosen categories within a date range.
    Returns: list of normalized paper dicts (deduped).
    """
    cats = categories or DEFAULT_CATEGORIES

    start_dt = _parse_yyyy_mm_dd(start_date)
    end_dt = _parse_yyyy_mm_dd(end_date)
    end_dt_inclusive = end_dt.replace(hour=23, minute=59)

    seen = set()
    out: List[Dict] = []

    headers = {
        "User-Agent": "nimbleminds-capstone-arxiv-fetcher/1.0 (contact: your_email@example.com)"
    }

    with httpx.Client(timeout=timeout, headers=headers) as client:
        window_start = start_dt

        while window_start <= end_dt_inclusive and len(out) < max_results_total:
            window_end = min(window_start + timedelta(days=chunk_days) - timedelta(minutes=1), end_dt_inclusive)

            date_from = _to_arxiv_dt(window_start)
            date_to = _to_arxiv_dt(window_end)
            query = _build_query(cats, date_from=date_from, date_to=date_to)

            start = start_offset
            while len(out) < max_results_total:
                batch = min(page_size, max_results_total - len(out))
                params = {
                    "search_query": query,
                    "start": start,
                    "max_results": batch,
                    "sortBy": "submittedDate",
                    "sortOrder": "descending",
                }

                xml = _get_with_retry(client, params=params)
                feed = feedparser.parse(xml)
                entries = feed.entries or []
                if not entries:
                    break

                for entry in entries:
                    paper = _parse_entry(entry)
                    key = paper.get("arxiv_id") or paper.get("id")
                    if key and key not in seen:
                        seen.add(key)
                        out.append(paper)

                if len(entries) < batch:
                    break

                start += batch
                if polite_delay_seconds:
                    time.sleep(polite_delay_seconds)

            print(f"[Fetcher] {window_start.date()} → {window_end.date()} | pulled so far: {len(out)}")
            if polite_delay_seconds:
                time.sleep(polite_delay_seconds)
            window_start = window_start + timedelta(days=chunk_days)

    print(f"[Fetcher] DONE | Total unique papers: {len(out)}")
    return out

# Saving fetched outputs (CSV only)
# -------------------------
def save_fetched_outputs(
    papers: List[Dict],
    start_date: str,
    end_date: str,
    base_folder: str = "Fetched",
    csv_name: str = "arxiv_fetched.csv",
) -> str:
    """
    Creates:
      Fetched/{start_date}:{end_date}/
        arxiv_fetched.csv

    Returns the output folder path.
    """
    run_folder = os.path.join(base_folder, f"{start_date}:{end_date}")
    os.makedirs(run_folder, exist_ok=True)

    if not papers:
        print(f"[Save] No papers to save in {run_folder}")
        return run_folder

    csv_path = os.path.join(run_folder, csv_name)

    # Union of all keys so we don't miss columns
    fieldnames = sorted({k for p in papers for k in p.keys()})

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(papers)

    print(f"[Save] Wrote {len(papers)} rows -> {csv_path}")
    return run_folder

# Example usage
# -------------------------
if __name__ == "__main__":
    start_date = "2023-01-01"
    end_date = "2023-01-31"

    papers = fetch_papers_by_date_range(
        start_date=start_date,
        end_date=end_date,
        max_results_total=25,
        chunk_days=1,
    )

    # Print a quick sanity check
    #for p in papers:
    #    primary_category = p.get("categories", ["unknown"])[0]
    #    section = primary_category.split(".")[0]
    #    print("\nSECTION:", section)
    #    print("CATEGORY:", primary_category)
    #    print("TITLE:", p.get("title", ""))
    #    print("ABSTRACT:", (p.get("abstract", "") or "")[:200])
    #    print("-" * 80)

[Fetcher] 2023-01-01 → 2023-01-01 | pulled so far: 25
[Fetcher] DONE | Total unique papers: 25


#2. Classifier

In [ ]:
from groq import Groq
# To clean the key and not have an error
raw_api_key = os.getenv("GROQ_API_KEY")
if raw_api_key is None:
    raise ValueError("GROQ_API_KEY is not set.")

clean_api_key = (
    str(raw_api_key)
    .replace("\u2028", "")
    .replace("\u2029", "")
    .replace("\n", "")
    .replace("\r", "")
    .strip()
)

client = Groq(api_key=clean_api_key)


print("Cleaned repr")
print("Length:", len(os.getenv("GROQ_API_KEY")))
print("Non-ASCII chars:", [(i, ch, hex(ord(ch))) for i, ch in enumerate(os.getenv("GROQ_API_KEY") or "") if ord(ch) > 127])

Cleaned repr
Length: 56
Non-ASCII chars: []


In [ ]:
import os
import re
import json
import time
import random
from typing import Dict, List, Tuple, Any, Optional
import unicodedata

from groq import Groq
from tqdm.auto import tqdm

# Groq client
# -------------------------
MODEL_NAME = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
client = Groq(api_key=clean_api_key)

# JSON parsing + helpers
# -------------------------
def _safe_json_load(s: str) -> Dict[str, Any]:
    """Try to parse JSON even if model adds extra text/code fences."""
    s = (s or "").strip()

    # Strip code fences
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\n?", "", s)
        s = s.replace("```", "").strip()

    # Attempt direct parse
    try:
        return json.loads(s)
    except Exception:
        pass

    # Try to extract first {...} block
    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1 and end > start:
        return json.loads(s[start : end + 1])

    raise ValueError(f"Could not parse JSON from model output:\n{s}")


def _truncate(text: str, max_chars: int) -> str:
    text = (text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


LLM_CALLS = 0  # shows up in progress bar

def _show_non_ascii(label: str, text: str):
    bad = [(i, ch, hex(ord(ch))) for i, ch in enumerate(text) if ord(ch) > 127]
    if bad:
        print(f"[NON-ASCII FOUND] {label}")
        for item in bad[:20]:
            print(item)

def _groq_json(prompt: str, max_attempts: int = 3, base_sleep: float = 1.0) -> Dict[str, Any]:
    global LLM_CALLS
    last_err: Optional[Exception] = None

    system_msg = _clean_text("Return STRICT JSON only. No markdown. No extra text.")
    user_msg = _clean_text(prompt)
    _show_non_ascii("SYSTEM MSG", system_msg)
    _show_non_ascii("USER MSG", user_msg)

    for attempt in range(1, max_attempts + 1):
        try:
            LLM_CALLS += 1
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg},
                ],
            )
            raw = _clean_text(resp.choices[0].message.content or "")
            return _safe_json_load(raw)
        except Exception as e:
            last_err = e
            time.sleep(base_sleep * (2 ** (attempt - 1)) + random.random() * 0.25)

    raise RuntimeError(f"Groq call failed after {max_attempts} attempts. Last error: {last_err}")

# 1) Keyword prefilter: Healthcare hints (*Recommend adding a few radiology/oncology/Pathology/general medicine terms to HEALTHCARE_HINTS*)
# -----------------------------------------------------
HEALTHCARE_HINTS = [
    "patient", "patients", "clinical", "clinician", "hospital", "diagnosis", "treatment",
    "disease", "screening", "prognosis", "therapy", "surgery", "mortality", "morbidity",
    "epidemiology", "public health", "cohort", "randomized", "trial", "ICU", "intensive care",
    "mri", "ct", "ultrasound", "x-ray", "radiology", "pathology",
    "ecg", "eeg", "ppg", "ehr", "electronic health record", "medical imaging",
]

HEALTHCARE_HINT_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in HEALTHCARE_HINTS) + r")\b",
    re.IGNORECASE
)


def keyword_healthcare_hits(title: str, abstract: str) -> List[str]:
    text = f"{title}\n{abstract}"
    return sorted(set(m.group(0) for m in HEALTHCARE_HINT_PATTERN.finditer(text)), key=str.lower)


# 2) Specialty keyword libraries (SCALABLE)
#   Outputs for now: Radiology, Oncology, Other
# ---------------------------------------------------------------------------------
RADIOLOGY_KEYWORDS = [
    "Radiology", "Radiologic imaging", "Radiological imaging", "Radiologist", "Diagnostic imaging",
    #"Medical imaging",
    "Radiography", "X-ray", "Computed tomography","Clinical Imaging",
    #"CT",
    "CT scan","CT scans", "Magnetic resonance imaging", "MRI", "MRI scan","MRI scans",
    "Ultrasound", "Ultrasonography", "Sonography", "Fluoroscopy",
    "Mammography", "CTA", "CT Angiography", "MR Angiography", "Interventional radiology",
    "Neuroradiology", "Musculoskeletal imaging", "Breast imaging",
    "Chest imaging", "Abdominal imaging", "Pelvic imaging", "Radiologists",
    "Thoracic imaging", "Cardiac imaging", "Vascular imaging",
    "Nuclear imaging", "Positron emission tomography", #"PET", "PET-CT",
    ## New words underneath
    "CT scan", "Computed tomography","PET scan", "PET imaging",
    "image segmentation", "image reconstruction", "radiomics","dicom", "imaging pipeline",
    "medical image analysis", "image analysis", "image segmentation", "deep learning for imaging",
    "organ segmentation", "tumor segmentation", "medical segmentation", "segmentation model",
    "segmentation models", "CT-based segmentation", "volumetric segmentation", "3D segmentation",
    "computed tomography imaging", "medical image segmentation",
    ## End of New words
    "Cross-sectional imaging", "Functional imaging", "Molecular imaging",
    "Imaging findings", "Radiologic findings", "Radiological findings",
    "Imaging modality", "Imaging modalities", "Imaging techniques",
    "Image interpretation", "Radiologic diagnosis", "Radiological evaluation",
    "Radiologic assessment", "Radiologist", "Screening imaging",
    "Tumor imaging", "Cancer imaging", "Brain imaging", "Spinal imaging",
    "Head and neck imaging", "Whole-body imaging", "Contrast-enhanced imaging",
    "Contrast media", "Image-guided diagnosis", "Image acquisition",
    "Image reconstruction", "Imaging biomarkers", "Diagnostic radiology",
    "Emergency radiology", "Pediatric radiology", "Body imaging", "Brain MRI", "Neuroimaging",
    "Diffusion-weighted imaging", "DWI",
    #NEWW
    "magnetic resonance", "magnetic resonance imaging",
    "image quality control", "quality control", "artifact", "artifacts",
]

ONCOLOGY_KEYWORDS = [
    "Oncology", "Cancer",# "Tumor", "Tumour", "Neoplasm", "Neoplasia",
    "Malignant tumor", "Solid tumor", "Hematologic malignancy", "Malignancy",
    "Carcinoma", "Sarcoma", "Lymphoma", "Leukemia", "Myeloma",
    "Metastasis", "Metastatic cancer", #"Primary tumor", "Secondary tumor", "Benign tumor",
    "Tumor progression", "Tumor growth", "Tumor burden", "Tumor heterogeneity",
    "Tumor microenvironment", "Cancer biology", "Cancer research",
    "Clinical oncology", "Medical oncology", "Surgical oncology", "Radiation oncology",
    "Pediatric oncology", "Translational oncology", "Precision oncology",
    "Molecular oncology", "Experimental oncology", "Breast cancer", "Lung cancer", "Prostate cancer", "Colorectal cancer",
    "Pancreatic cancer", "Liver cancer", "Hepatocellular carcinoma",
    "Gastric cancer", "Esophageal cancer", "Ovarian cancer", "Cervical cancer",
    "Endometrial cancer", "Bladder cancer", "Kidney cancer", "Renal cell carcinoma",
    "Brain tumor", "Glioma", "Glioblastoma", "skin cancer", #"Melanoma",
    "Thyroid cancer", "Head and neck cancer", "Chemotherapy", "Radiotherapy", "Radiation therapy",
    #"Targeted therapy","Immunotherapy", "Hormone therapy", "Endocrine therapy",
    "Cancer treatment", "Cancer therapy", #"Combination therapy",
    #"Adjuvant therapy", "Neoadjuvant therapy", "Palliative care",
    "CAR-T therapy", "Checkpoint inhibitors", #"PD-1", "PD-L1", "CTLA-4",
    #"Drug resistance",
    "Cancer drug development", "Cancer diagnosis", "Cancer screening", #"Early detection","Tumor detection",
    "Cancer prognosis",# "Survival analysis", "Disease progression",
    #"Recurrence", "Remission", "Clinical trials", "Phase I trial",
    #"Phase II trial", "Phase III trial", "Randomized controlled trial",
    "Oncology outcomes", "Oncogene", "Tumor suppressor gene",
    #"p53", "KRAS", "EGFR", "BRAF",
    "Cancer genomics", #"Genomic profiling", "Transcriptomics", "Proteomics",
    #"Biomarkers",
    "Tumor biomarkers", "Liquid biopsy",
    "Circulating tumor DNA", "ctDNA", "Cancer stem cells",
    #"Cell proliferation", "Apoptosis", "Angiogenesis",
    "Immune evasion", "Cancer incidence", "Cancer prevalence", "Cancer mortality", #"Risk factors",
    "Carcinogenesis", #"Environmental exposure", "Lifestyle factors",
    "Smoking-related cancer", "Cancer epidemiology", "Cancer disparities",
    "Global cancer burden", #"Biopsy", "Histopathology", "Cytology",
    "Tumor grading", "Tumor staging",
    "TNM staging", "Tumor markers", #"Disease monitoring","Minimal residual disease",
    "AI in oncology", "Machine learning cancer", "Deep learning oncology", "Immuno-oncology"
    #"Digital pathology",
    #"Multi-omics", "Epigenetics", "DNA methylation",
    ## New words added
    "neoadjuvant chemotherapy", "neoadjuvant chemoradiotherapy", "adjuvant chemotherapy", "adjuvant radiotherapy",
    "palliative chemotherapy",
]

PATHOLOGY_KEYWORDS= [
    "Biopsy", "Biopsies", "Core needle biopsy", "Breast biopsy", "Tissue biopsy",
    "Specimen analysis", "Tissue", "Tissue sample", "Whole slide imaging", "Whole slide images",
    "Slide imaging", "Glass slide", "Microscopy", "Digital microscopy", "Pathologist", "Pathologists",
    "Pathology report", "Histopathological analysis", "Histopathological evaluation", #"Diagnosis", "Diagnostic",
    #"Diagnostic accuracy", "Diagnostic assessment", "Diagnostic interpretation", "Tumor pathology",
    "Cancer pathology", "Lymphoma pathology", "Breast pathology", "Immunohistochemistry", "IHC", "Staining",
    "Tissue staining", "Cell morphology", "Morphological analysis", "Cellular features", "Tissue architecture",
    #"Image analysis",
    ## New words added
    "histology", "whole slide image", "WSI", "tissue section", "slide analysis",
    ## End of new words
    ##"Image classification", "Feature extraction", "Pattern recognition",
    "Artificial intelligence in pathology", "Machine learning pathology", "Deep learning pathology",
    #"Prediction", "Prognosis", "Risk stratification",
    "Workflow pathology", "Laboratory pathology", "Pathology workflow",
    # NEWW
    "clinical laboratory", "clinical laboratories", "laboratory medicine",
    "clinical pathology", "anatomic pathology", "lab test", "lab tests",
    "laboratory test", "laboratory tests", "specimen", "specimens",
    "reflex testing", "test ordering", "ferritin testing", "cbc testing",
]

DERMATOLOGY_HINTS = [
    "dermatology", "dermatologist", "dermatologists",
    "eczema", "psoriasis", "acne", "rosacea",
    "atopic dermatitis", "seborrheic dermatitis",
    "urticaria", "alopecia", "vitiligo"
]
#We can add more lists as we add more specialty keywords
# COLON_RECTAL_SURGERY_KEYWORDS = [ .... ]

SPECIALTY_KEYWORDS: Dict[str, List[str]] = {
    "Radiology": RADIOLOGY_KEYWORDS,
    "Oncology": ONCOLOGY_KEYWORDS,
    "Pathology": PATHOLOGY_KEYWORDS
    # Scalable as we can add more specialties over here
}

OUTPUT_SPECIALTIES = ["Radiology","Oncology","Pathology", "Other"]
#Add other specialties here

def _compile_specialty_patterns(spec_kw: Dict[str, List[str]]) -> Dict[str, re.Pattern]:
    patterns: Dict[str, re.Pattern] = {}
    for specialty, keywords in spec_kw.items():
        seen = set()
        deduped: List[str] = []
        for k in keywords:
            kk = (k or "").strip()
            if kk and kk.lower() not in seen:
                deduped.append(kk)
                seen.add(kk.lower())
        if deduped:
            patterns[specialty] = re.compile("|".join(re.escape(k) for k in deduped), re.IGNORECASE)
    return patterns

# DEBUG BLOCK
for specialty, keywords in SPECIALTY_KEYWORDS.items():
    print(f"\n--- {specialty} ---")
    print("Type of keywords object:", type(keywords))
    for i, k in enumerate(keywords):
        if not isinstance(k, str):
            print(f"Bad item at index {i}: {k} | type={type(k)}")


SPECIALTY_PATTERNS = _compile_specialty_patterns(SPECIALTY_KEYWORDS)


def score_specialty(title: str, abstract: str) -> Dict[str, Any]:
    """
    Keyword-only specialty classifier:
    - returns best specialty among the 3 if score>0
    - else "Other"
    Also returns matches for audit.
    """
    text = f"{title}\n{abstract}"
    text_lower = text.lower()

    # Dermatology papers should go to Other unless clearly cancer-related
    if any(k in text_lower for k in DERMATOLOGY_HINTS):
        if not any(k in text_lower for k in ["skin cancer", "cancer", "carcinoma"]):
            derm_matches = [k for k in DERMATOLOGY_HINTS if k in text_lower]
            return {
                "specialty": "Other",
                "keyword_score": len(derm_matches),
                "keyword_matches": derm_matches[:25],
            }

    best_spec = "Other"
    best_score = 0
    best_matches: List[str] = []

    for spec, pattern in SPECIALTY_PATTERNS.items():
        matches = sorted(set(m.group(0) for m in pattern.finditer(text)), key=str.lower)
        score = len(matches)
        if score > best_score:
            best_score = score
            best_spec = spec
            best_matches = matches

    return {
        "specialty": best_spec if best_score > 0 else "Other",
        "keyword_score": best_score,
        "keyword_matches": best_matches[:25],
    }

def _clean_text(text: Any) -> str:
    if text is None:
        return ""

    text = str(text)

    replacements = {
        "\u2028": " ",
        "\u2029": " ",
        "\xa0": " ",
        "–": "-",
        "—": "-",
        "“": '"',
        "”": '"',
        "’": "'",
        "‘": "'",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 3) LLM Healthcare Gate
# -------------------------
def llm_is_healthcare(title: str, abstract: str, kw_hits: List[str]) -> Dict[str, Any]:
    title = _clean_text(title)
    abstract = _clean_text(abstract)
    kw_hits = [_clean_text(k) for k in kw_hits]
    abstract_t = _truncate(abstract, 1400)

    prompt_lines = [
        "Label whether this arXiv paper is healthcare/medicine related.",
        "",
        "Definition of HEALTHCARE:",
        "- human health, clinical medicine, disease, diagnosis, treatment, patient care, healthcare systems,",
        "  public health/epidemiology, medical imaging used for diagnosis, medical devices used in care.",
        "",
        "NOT healthcare unless explicitly tied to human health/clinical context:",
        "- generic ML/optimization, robotics/control, pure neuroscience/cognition without clinical tie,",
        "- agriculture/plant biology/ecology, general biology without disease/clinical relevance.",
        "",
        f"Keyword hints found (may help, but do not over-trust): {json.dumps(kw_hits)}",
        "",
        "Return STRICT JSON:",
        "{",
        '  "is_healthcare": true,',
        '  "confidence": 0.0,',
        '  "reason": "short evidence-based reason referencing phrases from title/abstract"',
        "}",
        "",
        f"TITLE: {title}",
        f"ABSTRACT: {abstract_t}",
    ]
    _show_non_ascii("TITLE", title)
    _show_non_ascii("ABSTRACT", abstract_t)
    prompt = _clean_text("\n".join(prompt_lines))
    data = _groq_json(prompt)

    return {
        "is_healthcare": bool(data.get("is_healthcare", False)),
        "confidence": float(data.get("confidence", 0.0)),
        "reason": _clean_text(str(data.get("reason", "")).strip()),
    }

# 4) Full classifier for one paper (scalable)
# -----------------------------------------------------
def classify_paper(paper: Dict[str, Any], llm_threshold: float = 0.55) -> Dict[str, Any]:
    """
    Expects paper fields from your fetcher:
      - title
      - abstract   (IMPORTANT: your fetcher uses "abstract")
      - categories (list)
      - plus any metadata (arxiv_id, link, pdf_url, ...)
    """
    title = _clean_text(paper.get("title", "") or "")
    abstract = _clean_text(paper.get("abstract", "") or "")

    kw_hits = keyword_healthcare_hits(title, abstract)

    # Optimization: if zero hints, skip LLM to save calls (conservative)
    if not kw_hits:
        hc = {
            "is_healthcare": False,
            "confidence": 0.35,
            "reason": "No healthcare keyword hints; skipped LLM gate.",
        }
    else:
        hc = llm_is_healthcare(title, abstract, kw_hits)
        # Optional conservative threshold
        if hc["is_healthcare"] and hc["confidence"] < llm_threshold:
            hc["is_healthcare"] = False
            hc["reason"] = f"Low-confidence healthcare ({hc['confidence']}); treated as non-healthcare. " + hc["reason"]

    # Specialty only if healthcare
    if hc["is_healthcare"]:
        spec = score_specialty(title, abstract)
    else:
        spec = {"specialty": "Other", "keyword_score": 0, "keyword_matches": []}

    out = dict(paper)
    out.update({
        "healthcare": hc["is_healthcare"],
        "hc_confidence": hc["confidence"],
        "hc_reason": hc["reason"],
        "hc_keyword_hits": "|".join(kw_hits[:25]),
        "specialty": spec["specialty"],
        "spec_keyword_score": spec["keyword_score"],
        "spec_keyword_matches": "|".join(spec["keyword_matches"]),
    })
    return out


# 5) Batch runner + progress bar + summary
# -----------------------------------------
def classify_batch(papers: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    total = len(papers)
    hc_count = 0
    spec_counts: Dict[str, int] = {s: 0 for s in OUTPUT_SPECIALTIES}

    bar = tqdm(papers, desc="Classifying papers", unit="paper")
    for i, p in enumerate(bar, start=1):
        try:
            row = classify_paper(p)
            rows.append(row)

            if row["healthcare"]:
                hc_count += 1
                spec = row["specialty"]
                if spec not in spec_counts:
                    spec = "Other"
                spec_counts[spec] += 1

            bar.set_postfix(llm_calls=LLM_CALLS, healthcare=hc_count)

        except Exception as e:
            print(f"\n[ERROR] Failed on paper #{i}")
            print("RAW TITLE:", repr(p.get("title", "")))
            print("RAW ABSTRACT:", repr((p.get("abstract", "") or "")[:500]))
            raise

    summary = {
        "total": total,
        "healthcare": hc_count,
        "healthcare_rate": (hc_count / total) if total else 0.0,
        "specialties": spec_counts,
    }
    return rows, summary


--- Radiology ---
Type of keywords object: <class 'list'>

--- Oncology ---
Type of keywords object: <class 'list'>

--- Pathology ---
Type of keywords object: <class 'list'>


In [ ]:
# To share the folders to our google drive, we need to give permission to access
# run the code below to give that access
# Make the path for the shared AI file on your drive to look similar to this below
# content/drive/MyDrive/AI/...

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys

MYDRIVE = "/content/drive/MyDrive"

AI_ROOT = None
for item in os.listdir(MYDRIVE):
    if item.startswith("AI"):
        candidate = os.path.join(MYDRIVE, item)

        # CHANGED: now checking inside "2025 Team/Scraping Agent"
        scraping_agent_path = os.path.join(candidate, "2025 Team", "Scraping Agent")

        if os.path.isdir(candidate) and os.path.exists(scraping_agent_path):
            AI_ROOT = candidate
            break

if AI_ROOT is None:
    raise FileNotFoundError("Could not find shared AI folder in MyDrive.")

print("Using AI root:", AI_ROOT)

# CHANGED: added "2025 Team" into the path
TEAM_DIR = os.path.join(AI_ROOT, "2025 Team")

BASE_DIR = os.path.join(TEAM_DIR, "Scraping Agent", "Output")
FETCHED_DIR = os.path.join(BASE_DIR, "Fetched")
CLASSIFIED_DIR = os.path.join(BASE_DIR, "Classified")

REFERENCE_XLSX_PATH = os.path.join(
    TEAM_DIR,
    "Datasets",
    "Copy of Proprietary Training Data Summary - Running Total v2.xlsx"
)

os.makedirs(FETCHED_DIR, exist_ok=True)
os.makedirs(CLASSIFIED_DIR, exist_ok=True)

print("Reference file exists:", os.path.exists(REFERENCE_XLSX_PATH))
print("Fetched dir:", FETCHED_DIR)
print("Classified dir:", CLASSIFIED_DIR)

Using AI root: /content/drive/MyDrive/AI
Reference file exists: True
Fetched dir: /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Fetched
Classified dir: /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified


In [ ]:
#import os

#BASE_DIR = "/content/drive/MyDrive/AI/Scraping Agent/Output"

#FETCHED_DIR = os.path.join(BASE_DIR, "Fetched")
#CLASSIFIED_DIR = os.path.join(BASE_DIR, "Classified")

#os.makedirs(FETCHED_DIR, exist_ok=True)
#os.makedirs(CLASSIFIED_DIR, exist_ok=True)

#print("Created folders:")
#print(FETCHED_DIR)
#print(CLASSIFIED_DIR)

# 3. Storage

In [ ]:
import os
import csv
from collections import defaultdict
from typing import List, Dict, Any

def save_rows_to_csv(rows: List[Dict[str, Any]], path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if not rows:
        print(f"[Save] No rows to save -> {path}")
        return False

    fieldnames = sorted({k for r in rows for k in r.keys()})

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0

    if not exists or size == 0:
        raise RuntimeError(f"File write failed or empty file: {path}")

    print(f"[Save] Wrote {len(rows)} rows -> {path} (exists={exists}, size={size})")
    return True

def save_rows_to_csv_allow_empty(rows: List[Dict[str, Any]], path: str, fieldnames: List[str]):
    """
    Saves a CSV even when rows is empty.
    This is useful for useful_only files, where 0 useful rows is still a valid result.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        if rows:
            writer.writerows(rows)

    print(f"[Save] Wrote {len(rows)} rows -> {path}")
    return True

def _slugify(name: str) -> str:
    import re
    name = (name or "").strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_") or "other"


from collections import defaultdict
import os

def save_specialty_outputs(
    rows: List[Dict[str, Any]],
    base_dir: str,
    start_date: str = "",
    end_date: str = "",
):
    os.makedirs(base_dir, exist_ok=True)
    classified_rows = [
        r for r in rows
        if r.get("healthcare")
        and str(r.get("specialty", "")).strip() in {"Oncology", "Pathology", "Radiology", "Other"}
    ]
    print(f"[Debug] classified_rows = {len(classified_rows)}")

    # Save master classified file
    master_path = os.path.join(base_dir, f"{start_date}_{end_date}_specialties_master.csv")
    save_rows_to_csv(classified_rows, master_path)

    # Save one file per specialty
    buckets = defaultdict(list)
    for r in classified_rows:
        specialty = r.get("specialty", "Other")
        buckets[specialty].append(r)

    for specialty, spec_rows in buckets.items():
        print(f"[Debug] Saving specialty = {specialty}, rows = {len(spec_rows)}")

        folder = os.path.join(base_dir, _slugify(specialty))
        os.makedirs(folder, exist_ok=True)

        out_path = os.path.join(folder, f"{start_date}_{end_date}_{_slugify(specialty)}.csv")
        save_rows_to_csv(spec_rows, out_path)

def save_run_summary(summary: Dict[str, Any], output_path: str):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k, v in summary.items():
            writer.writerow([k, v])

    print(f"[Save] Wrote summary -> {output_path}")

# helper so that after classification, loop through the healthcare rows and extract full text
def save_full_texts_by_specialty(rows: List[Dict[str, Any]], base_dir: str):
    for row in rows:
        if not row.get("healthcare"):
            continue

        if already_saved_in_specialty(row, base_dir):
            print(f"[Skip] Already saved full text for {row.get('arxiv_id')}")
            continue

        pdf_url = row.get("pdf_url", "")
        if not pdf_url:
            print(f"[Skip] No PDF URL for {row.get('arxiv_id')}")
            continue

        try:
            full_text = extract_text_from_pdf_url(pdf_url)
            abstract_pdf = extract_abstract_from_pdf_url(pdf_url)

            row["abstract_pdf"] = abstract_pdf

            append_full_text_to_specialty_file(row, full_text, base_dir)
            mark_saved_in_specialty(row, base_dir)
        except Exception as e:
            print(f"[Error] Failed full-text extraction for {row.get('arxiv_id')}: {e}")

In [ ]:
## Helper to extract from PDF
import fitz  # PyMuPDF
import httpx
import os

def extract_text_from_pdf_url(pdf_url: str, temp_pdf_path: str = "/tmp/temp_paper.pdf") -> str:
    if not pdf_url:
        return ""

    r = httpx.get(pdf_url, timeout=60)
    r.raise_for_status()

    with open(temp_pdf_path, "wb") as f:
        f.write(r.content)

    text_parts = []
    doc = fitz.open(temp_pdf_path)
    for page in doc:
        text_parts.append(page.get_text())
    doc.close()

    return "\n".join(text_parts).strip()

## Helper to extract from PDF
import fitz  # PyMuPDF
import httpx
import os

def extract_text_from_pdf_url(pdf_url: str, temp_pdf_path: str = "/tmp/temp_paper.pdf") -> str:
    if not pdf_url:
        return ""

    r = httpx.get(pdf_url, timeout=60)
    r.raise_for_status()

    with open(temp_pdf_path, "wb") as f:
        f.write(r.content)

    text_parts = []
    doc = fitz.open(temp_pdf_path)
    for page in doc:
        text_parts.append(page.get_text())
    doc.close()

    return "\n".join(text_parts).strip()

import re
import fitz
import httpx

def _clean_pdf_text(text: str) -> str:
    if not text:
        return ""

    # fix hyphenation across line breaks
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)

    # normalize whitespace
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()


def _page_text_in_reading_order(page) -> str:
    """
    Extract text blocks and sort them in reading order:
    top-to-bottom, then left-to-right.
    This is more reliable than page.get_text() for multi-column PDFs.
    """
    blocks = page.get_text("blocks")
    # block tuple: (x0, y0, x1, y1, text, block_no, block_type, ...)
    text_blocks = [b for b in blocks if len(b) >= 5 and isinstance(b[4], str) and b[4].strip()]
    text_blocks.sort(key=lambda b: (round(b[1], 1), round(b[0], 1)))
    return "\n".join(b[4].strip() for b in text_blocks)


def _extract_abstract_from_text(full_text: str) -> str:
    if not full_text:
        return ""

    text = _clean_pdf_text(full_text)

    # Start at Abstract / ABSTRACT / Abstract—
    start_match = re.search(r"\bAbstract\b\s*[\.\-—:]?\s*", text, flags=re.IGNORECASE)
    if not start_match:
        return ""

    text = text[start_match.end():]

    # Stop at first major section heading, but DO NOT stop at
    # Objective / Approach / Main Results / Significance
    end_patterns = [
        r"\n\s*Keywords\b",
        r"\n\s*Index Terms\b",
        r"\n\s*I\.\s*INTRODUCTION\b",
        r"\n\s*1\.\s*Introduction\b",
        r"\n\s*1\s+Introduction\b",
        r"\n\s*Introduction\b",
        r"\n\s*II\.\s*",
        r"\n\s*2\.\s*",
    ]

    end_positions = []
    for pattern in end_patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            end_positions.append(m.start())

    if end_positions:
        text = text[:min(end_positions)]

    text = _clean_pdf_text(text)

    # Final flattening for one-paragraph abstract storage
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_abstract_from_pdf_url(
    pdf_url: str,
    temp_pdf_path: str = "/tmp/temp_paper.pdf",
    max_pages: int = 2,
) -> str:
    if not pdf_url:
        return ""

    r = httpx.get(pdf_url, timeout=60)
    r.raise_for_status()

    with open(temp_pdf_path, "wb") as f:
        f.write(r.content)

    doc = fitz.open(temp_pdf_path)
    try:
        pages_text = []
        for i, page in enumerate(doc):
            if i >= max_pages:
                break
            pages_text.append(_page_text_in_reading_order(page))

        first_pages_text = "\n\n".join(pages_text)
        return _extract_abstract_from_text(first_pages_text)
    finally:
        doc.close()

# Helper to append article text to specialty
def append_full_text_to_specialty_file(
    paper: Dict[str, Any],
    full_text: str,
    base_dir: str,
):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    txt_path = os.path.join(folder, f"{folder_name}_full_articles.txt")

    with open(txt_path, "a", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write(f"ARXIV ID: {paper.get('arxiv_id', '')}\n")
        f.write(f"TITLE: {paper.get('title', '')}\n")
        f.write(f"SPECIALTY: {specialty}\n")
        f.write(f"PDF URL: {paper.get('pdf_url', '')}\n")
        f.write(f"AUTHORS: {', '.join(paper.get('authors', []))}\n")
        f.write(f"PUBLISHED: {paper.get('published', '')}\n")
        f.write(f"ABSTRACT PDF: {paper.get('abstract_pdf', '')}\n")
        f.write("=" * 60 + "\n\n")
        f.write("FULL TEXT:\n")
        f.write(full_text or "[NO TEXT EXTRACTED]")
        f.write("\n\nEND OF ARTICLE\n")
        f.write("#" * 60 + "\n\n")

    print(f"[Save] Appended article text -> {txt_path}")

# Helper to avoid duplicates
def already_saved_in_specialty(paper: Dict[str, Any], base_dir: str) -> bool:
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if not arxiv_id:
        return False

    if os.path.exists(id_log_path):
        with open(id_log_path, "r", encoding="utf-8") as f:
            existing = set(line.strip() for line in f if line.strip())
        return arxiv_id in existing

    return False

def mark_saved_in_specialty(paper: Dict[str, Any], base_dir: str):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if arxiv_id:
        with open(id_log_path, "a", encoding="utf-8") as f:
            f.write(arxiv_id + "\n")

# Helper to append article text to specialty
def append_full_text_to_specialty_file(
    paper: Dict[str, Any],
    full_text: str,
    base_dir: str,
):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    txt_path = os.path.join(folder, f"{folder_name}_full_articles.txt")

    with open(txt_path, "a", encoding="utf-8") as f:
        f.write("=" * 60 + "\n")
        f.write(f"ARXIV ID: {paper.get('arxiv_id', '')}\n")
        f.write(f"TITLE: {paper.get('title', '')}\n")
        f.write(f"SPECIALTY: {specialty}\n")
        f.write(f"PDF URL: {paper.get('pdf_url', '')}\n")
        f.write(f"AUTHORS: {', '.join(paper.get('authors', []))}\n")
        f.write(f"PUBLISHED: {paper.get('published', '')}\n")
        f.write(f"ABSTRACT PDF: {paper.get('abstract_pdf', '')}\n")
        f.write("=" * 60 + "\n\n")
        f.write("FULL TEXT:\n")
        f.write(full_text or "[NO TEXT EXTRACTED]")
        f.write("\n\nEND OF ARTICLE\n")
        f.write("#" * 60 + "\n\n")

    print(f"[Save] Appended article text -> {txt_path}")

# Helper to avoid duplicates
def already_saved_in_specialty(paper: Dict[str, Any], base_dir: str) -> bool:
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if not arxiv_id:
        return False

    if os.path.exists(id_log_path):
        with open(id_log_path, "r", encoding="utf-8") as f:
            existing = set(line.strip() for line in f if line.strip())
        return arxiv_id in existing

    return False


def mark_saved_in_specialty(paper: Dict[str, Any], base_dir: str):
    specialty = paper.get("specialty", "Other")
    folder_name = _slugify(specialty)
    folder = os.path.join(base_dir, folder_name)
    os.makedirs(folder, exist_ok=True)

    id_log_path = os.path.join(folder, f"{folder_name}_saved_ids.txt")
    arxiv_id = paper.get("arxiv_id", "")

    if arxiv_id:
        with open(id_log_path, "a", encoding="utf-8") as f:
            f.write(arxiv_id + "\n")

In [ ]:
import os
import re
import httpx
import fitz
from urllib.parse import urlparse
from typing import List, Dict, Any, Optional

# =========================================================
# METADATA + DATASET +IMAGE + EXTRACTION HELPERS
# =========================================================

DATASET_DOMAINS = [
    "physionet.org",
    "kaggle.com",
    "zenodo.org",
    "figshare.com",
    "datadryad.org",
    "dryad.org",
    "osf.io",
    "huggingface.co",
    "openml.org",
    "archive.ics.uci.edu",
    "openneuro.org",
    "cancerimagingarchive.net",
    "brain-development.org",
    "ncbi.nlm.nih.gov",
    "bioproject.ncbi.nlm.nih.gov",
    "nih.gov",
    "cdc.gov",
    "who.int",
    "data.gov",
]

DATASET_HINT_PATTERNS = [
    r"\bdataset\b",
    r"\bdata set\b",
    r"\bpublic dataset\b",
    r"\bpublicly available dataset\b",
    r"\brepository\b",
    r"\bregistry\b",
    r"\bbiobank\b",
    r"\bcohort\b",
    r"\bdata source\b",
    r"\bavailable at\b",
    r"\bdownloaded from\b",
    r"\bobtained from\b",
    r"\baccessed from\b",
]

URL_PATTERN = re.compile(r'https?://[^\s<>"\)\]]+', re.IGNORECASE)


def clean_url(url: str) -> str:
    return (url or "").strip().rstrip(".,;:)")


def is_dataset_like_url(url: str) -> bool:
    try:
        parsed = urlparse(url)
        netloc = parsed.netloc.lower()
        path = parsed.path.lower()
        full = f"{netloc}{path}"

        if any(domain in netloc for domain in DATASET_DOMAINS):
            return True

        if "github.com" in netloc or "gitlab.com" in netloc:
            good_terms = ["dataset", "datasets", "data", "corpus", "benchmark"]
            bad_terms = [
                "model", "models", "tool", "tools", "framework",
                "train", "training", "inference", "code",
                "implementation", "demo", "seq2seq"
            ]
            has_good = any(term in full for term in good_terms)
            has_bad = any(term in full for term in bad_terms)
            return has_good and not has_bad

        fallback_terms = ["dataset", "download", "accession", "cohort", "registry"]
        return any(term in full for term in fallback_terms)

    except Exception:
        return False


def extract_dataset_links_from_text(text: str) -> List[str]:
    """
    Extracts ALL URLs from the article PDF text.
    This is intentionally broad because raw CSV should contain every link.
    """
    urls = [clean_url(m.group(0)) for m in URL_PATTERN.finditer(text or "")]
    urls = list(dict.fromkeys(urls))

    return urls


def extract_dataset_mentions(text: str, window: int = 120) -> List[str]:
    text = re.sub(r"\s+", " ", text or "").strip()
    mentions = []

    for pattern in DATASET_HINT_PATTERNS:
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            start = max(0, m.start() - window)
            end = min(len(text), m.end() + window)
            snippet = text[start:end].strip()
            mentions.append(snippet)

    unique = []
    seen = set()
    for snippet in mentions:
        key = snippet.lower()
        if key not in seen:
            seen.add(key)
            unique.append(snippet)

    return unique


def extract_dataset_info_from_pdf_url(pdf_url: str) -> Dict[str, Any]:
    text = extract_text_from_pdf_url(pdf_url)

    dataset_links = extract_dataset_links_from_text(text)
    dataset_mentions = extract_dataset_mentions(text)

    return {
        "dataset_links": dataset_links,
        "dataset_mentions": dataset_mentions,
    }
#Main metadata

# =========================================================
# IMAGE EXTRACTION
# =========================================================

import os
import io
import re
import csv
import hashlib
import tempfile
from typing import List, Dict

import httpx
import fitz
from PIL import Image

def add_dataset_info_to_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Adds dataset links and dataset mentions to each healthcare row.
    Uses the existing extract_dataset_info_from_pdf_url() function.
    """
    for row in rows:
        if not row.get("healthcare"):
            row["dataset_links"] = ""
            row["dataset_mentions"] = ""
            row["dataset_link_count"] = 0
            continue

        pdf_url = row.get("pdf_url", "")

        if not pdf_url:
            row["dataset_links"] = ""
            row["dataset_mentions"] = ""
            row["dataset_link_count"] = 0
            print(f"[Datasets] Skip {row.get('arxiv_id')} -> no PDF URL")
            continue

        try:
            dataset_info = extract_dataset_info_from_pdf_url(pdf_url)

            dataset_links = dataset_info.get("dataset_links", [])
            dataset_mentions = dataset_info.get("dataset_mentions", [])

            row["dataset_links"] = " | ".join(dataset_links)
            row["dataset_mentions"] = " | ".join(dataset_mentions[:10])
            row["dataset_link_count"] = len(dataset_links)

            print(
                f"[Datasets] {row.get('arxiv_id')} -> "
                f"{len(dataset_links)} links, {len(dataset_mentions)} mentions"
            )

        except Exception as e:
            row["dataset_links"] = ""
            row["dataset_mentions"] = ""
            row["dataset_link_count"] = 0
            row["dataset_error"] = str(e)
            print(f"[Datasets] Failed for {row.get('arxiv_id')}: {e}")

    return rows



def clean_caption_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def extract_figure_label(text: str) -> str:
    m = re.search(
        r"\b(Supplementary\s+Fig(?:ure)?\.?\s*\d+[A-Za-z]?|Fig(?:ure)?\.?\s*\d+[A-Za-z]?)\b",
        text,
        flags=re.IGNORECASE,
    )
    return m.group(1) if m else ""


def safe_caption_slug(text: str, max_len: int = 100) -> str:
    text = str(text or "").lower().strip()
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^a-z0-9_]+", "", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:max_len] if text else ""


def truncate_caption_for_filename(text: str, max_words: int = 12) -> str:
    """
    Shortens a caption before slugging so filenames don't get too long.
    """
    text = clean_caption_text(text)
    if not text:
        return ""
    words = text.split()
    return " ".join(words[:max_words])


def find_nearby_caption(page, img_rect, max_distance: float = 180) -> str:
    """
    Find the most likely caption near an image rectangle.
    Preference:
      1) below image
      2) above image
      3) text with Figure/Fig pattern
      4) text with some horizontal overlap or near-horizontal alignment
    """
    blocks = page.get_text("blocks")
    candidates = []

    for block in blocks:
        x0, y0, x1, y1, text = block[:5]
        text = clean_caption_text(text)
        if not text:
            continue

        caption_like = bool(
            re.search(
                r"\b(Supplementary\s+Fig(?:ure)?\.?\s*\d+|Fig(?:ure)?\.?\s*\d+)\b",
                text,
                re.IGNORECASE,
            )
        )

        horizontal_overlap = max(0, min(img_rect.x1, x1) - max(img_rect.x0, x0))
        horizontal_gap = min(abs(x0 - img_rect.x1), abs(x1 - img_rect.x0))

        # text below image
        if y0 >= img_rect.y1:
            vertical_gap = y0 - img_rect.y1
            if vertical_gap <= max_distance and (horizontal_overlap > 0 or horizontal_gap < 100):
                score = vertical_gap
                if horizontal_overlap <= 0:
                    score += 30
                if caption_like:
                    score -= 40
                candidates.append((score, text))

        # text above image
        elif y1 <= img_rect.y0:
            vertical_gap = img_rect.y0 - y1
            if vertical_gap <= max_distance and (horizontal_overlap > 0 or horizontal_gap < 100):
                score = vertical_gap + 20
                if horizontal_overlap <= 0:
                    score += 30
                if caption_like:
                    score -= 40
                candidates.append((score, text))

    if not candidates:
        return ""

    candidates.sort(key=lambda x: x[0])
    return candidates[0][1]


def extract_images_from_pdf_url(
    pdf_url: str,
    output_dir: str,
    min_width: int = 160,
    min_height: int = 160,
    min_bytes: int = 8000,
    max_images_per_page: int = 20,
    save_metadata_csv: bool = True,
) -> List[Dict]:
    """
    Extract embedded images from a PDF URL, capture nearby captions,
    and rename saved files using the caption when available.

    Returns a list of metadata dicts, one per saved image.
    """
    os.makedirs(output_dir, exist_ok=True)

    response = httpx.get(pdf_url, timeout=60, follow_redirects=True)
    response.raise_for_status()
    pdf_bytes = response.content

    saved_items = []
    seen_hashes = set()

    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
        tmp.write(pdf_bytes)
        temp_pdf = tmp.name

    doc = None
    try:
        doc = fitz.open(temp_pdf)

        for page_num in range(len(doc)):
            page = doc[page_num]
            image_list = page.get_images(full=True)

            if not image_list:
                continue

            for img_index, img in enumerate(image_list[:max_images_per_page]):
                xref = img[0]

                try:
                    base_image = doc.extract_image(xref)
                    image_bytes = base_image["image"]
                    ext = (base_image.get("ext") or "png").lower()
                    width = int(base_image.get("width", 0) or 0)
                    height = int(base_image.get("height", 0) or 0)

                    if width < min_width or height < min_height:
                        continue
                    if len(image_bytes) < min_bytes:
                        continue

                    img_hash = hashlib.md5(image_bytes).hexdigest()
                    if img_hash in seen_hashes:
                        continue
                    seen_hashes.add(img_hash)

                    pil_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

                    rects = page.get_image_rects(xref)
                    best_rect = rects[0] if rects else None

                    caption = ""
                    figure_label = ""
                    bbox = ""

                    if best_rect is not None:
                        caption = find_nearby_caption(page, best_rect)
                        figure_label = extract_figure_label(caption)
                        bbox = f"{best_rect.x0:.1f},{best_rect.y0:.1f},{best_rect.x1:.1f},{best_rect.y1:.1f}"

                    out_ext = "jpg" if ext in {"jpeg", "jpg"} else "png"

                    # Prefer figure label first, otherwise use shortened caption
                    filename_caption_source = figure_label or truncate_caption_for_filename(caption)
                    caption_slug = safe_caption_slug(filename_caption_source)

                    if caption_slug:
                        filename = f"page_{page_num+1}_img_{img_index+1}_{caption_slug}.{out_ext}"
                    else:
                        filename = f"page_{page_num+1}_img_{img_index+1}.{out_ext}"

                    filepath = os.path.join(output_dir, filename)

                    # avoid accidental filename collisions
                    counter = 2
                    original_filepath = filepath
                    original_filename = filename
                    while os.path.exists(filepath):
                        stem, ext2 = os.path.splitext(original_filename)
                        filename = f"{stem}_{counter}{ext2}"
                        filepath = os.path.join(output_dir, filename)
                        counter += 1

                    if out_ext == "jpg":
                        pil_img.save(filepath, format="JPEG", quality=95)
                    else:
                        pil_img.save(filepath, format="PNG")

                    item = {
                        "page_num": page_num + 1,
                        "image_index": img_index + 1,
                        "xref": xref,
                        "saved_filename": filename,
                        "saved_path": filepath,
                        "width": width,
                        "height": height,
                        "caption": caption,
                        "figure_label": figure_label,
                        "bbox": bbox,
                    }
                    saved_items.append(item)

                except Exception as e:
                    print(f"[Image Extract] Could not extract image on page {page_num+1}, image {img_index+1}: {e}")

        if save_metadata_csv and saved_items:
            csv_path = os.path.join(output_dir, "image_metadata.csv")
            with open(csv_path, "w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=saved_items[0].keys())
                writer.writeheader()
                writer.writerows(saved_items)

    finally:
        if doc is not None:
            doc.close()
        try:
            os.remove(temp_pdf)
        except Exception:
            pass

    return saved_items

In [ ]:
import os
import re
import unicodedata
from typing import List, Dict, Any


def safe_folder_name(name: str, max_len: int = 120) -> str:
    name = unicodedata.normalize("NFKD", str(name or "")).encode("ascii", "ignore").decode("ascii")
    name = re.sub(r'[\\/*?:"<>|]', "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name[:max_len] if name else "untitled_paper"


def save_image_assets(rows: List[Dict[str, Any]], base_dir: str):
    """
    Image-only asset saver.
    Leaves dataset / metadata code untouched.
    """
    healthcare_rows = [r for r in rows if r.get("healthcare")]

    for row in healthcare_rows:
        specialty = row.get("specialty", "Other")
        specialty_folder = _slugify(specialty)

        arxiv_id = row.get("arxiv_id", "unknown_id")
        title = row.get("title", "untitled_paper")
        pdf_url = row.get("pdf_url", "")

        paper_folder = safe_folder_name(
            f"{arxiv_id}_{title}"
        )

        specialty_dir = os.path.join(base_dir, specialty_folder)
        metadata_dir = os.path.join(specialty_dir, "Metadata")
        images_root_dir = os.path.join(metadata_dir, "images")
        paper_image_dir = os.path.join(images_root_dir, paper_folder)

        os.makedirs(metadata_dir, exist_ok=True)
        os.makedirs(images_root_dir, exist_ok=True)
        os.makedirs(paper_image_dir, exist_ok=True)

        try:
            existing_files = [
                f for f in os.listdir(paper_image_dir)
                if os.path.isfile(os.path.join(paper_image_dir, f))
                and f.lower().endswith((".png", ".jpg", ".jpeg"))
            ]

            if existing_files:
                print(f"[Images] Skip {row.get('arxiv_id')} -> images already exist")
            else:
                saved = extract_images_from_pdf_url(
                    pdf_url=pdf_url,
                    output_dir=paper_image_dir,
                )
                print(f"[Images] Saved {len(saved)} images for {row.get('arxiv_id')}")

        except Exception as e:
            print(f"[Images] Extraction failed for {row.get('arxiv_id')}: {e}")

In [ ]:
import os
import re
import unicodedata
import pandas as pd
from difflib import SequenceMatcher
from urllib.parse import urlparse, unquote

#REFERENCE_XLSX_PATH = "/content/drive/AI/Datasets/Copy of Proprietary Training Data Summary - Running Total v2.xlsx"
REFERENCE_SHEETS = ["Data List", "Data List (cornell)"]


def normalize_text(s: str) -> str:
    s = str(s or "")
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower().strip()
    s = s.replace(".zip", " ").replace(".tar.gz", " ").replace(".tgz", " ").replace(".csv", " ")
    s = re.sub(r"[_/\-]+", " ", s)
    s = re.sub(r"[^a-z0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


STOPWORDS = {
    "dataset", "data", "images", "image", "study", "paper", "public", "available",
    "the", "and", "for", "with", "from", "using", "used", "zip"
}


def text_to_tokens(s: str) -> set:
    s = normalize_text(s)
    return {tok for tok in s.split() if len(tok) > 2 and tok not in STOPWORDS}


def overlap_score(a_tokens: set, b_tokens: set) -> float:
    if not a_tokens or not b_tokens:
        return 0.0
    inter = len(a_tokens & b_tokens)
    return inter / max(1, min(len(a_tokens), len(b_tokens)))


def similarity_ratio(a: str, b: str) -> float:
    a = normalize_text(a)
    b = normalize_text(b)
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def get_url_tail(url: str) -> str:
    try:
        path = unquote(urlparse(url).path)
        tail = os.path.basename(path)
        return normalize_text(tail)
    except Exception:
        return ""


def load_reference_catalog(xlsx_path: str = REFERENCE_XLSX_PATH) -> pd.DataFrame:
    frames = []

    for sheet_name in REFERENCE_SHEETS:
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name)

        needed_cols = [
            "File Name",
            "Broad Group",
            "Specialty",
            "Modality in Bucket",
            "Nimblemind/Customer/Non-Commercial",
        ]

        for col in needed_cols:
            if col not in df.columns:
                df[col] = ""

        df = df[needed_cols].copy()
        df["source_sheet"] = sheet_name
        df = df[df["File Name"].notna()].copy()
        df["File Name"] = df["File Name"].astype(str).str.strip()
        df = df[df["File Name"] != ""]

        df["ref_name_norm"] = df["File Name"].apply(normalize_text)
        df["ref_tokens"] = (
            df["File Name"].fillna("").astype(str) + " " +
            df["Broad Group"].fillna("").astype(str) + " " +
            df["Specialty"].fillna("").astype(str) + " " +
            df["Modality in Bucket"].fillna("").astype(str)
        ).apply(text_to_tokens)

        frames.append(df)

    ref_df = pd.concat(frames, ignore_index=True).drop_duplicates(
        subset=["source_sheet", "File Name"]
    ).reset_index(drop=True)

    return ref_df


REFERENCE_CATALOG = load_reference_catalog()


def classify_dataset_usefulness(
    row: dict,
    dataset_link: str,
    dataset_mentions,
    ref_df: pd.DataFrame = REFERENCE_CATALOG
) -> dict:
    dataset_mentions = dataset_mentions or []
    mention_text = " ".join(dataset_mentions[:5]) if isinstance(dataset_mentions, list) else str(dataset_mentions)

    title = row.get("title", "")
    abstract = row.get("abstract", "")
    specialty = row.get("specialty", "")
    pdf_url = row.get("pdf_url", "")

    candidate_text = " ".join([
        title,
        abstract,
        specialty,
        dataset_link or "",
        mention_text,
        pdf_url,
    ])

    candidate_norm = normalize_text(candidate_text)
    candidate_tokens = text_to_tokens(candidate_text)

    link_tail = get_url_tail(dataset_link or "")
    paper_specialty_tokens = text_to_tokens(specialty)

    best = None
    best_score = -1

    for _, ref in ref_df.iterrows():
        ref_name = ref["File Name"]
        ref_name_norm = ref["ref_name_norm"]
        ref_tokens = ref["ref_tokens"]

        name_score = max(
            similarity_ratio(link_tail, ref_name_norm),
            similarity_ratio(candidate_norm, ref_name_norm) if len(ref_name_norm) < 40 else 0.0
        )

        token_score = overlap_score(candidate_tokens, ref_tokens)

        ref_specialty_tokens = text_to_tokens(ref.get("Specialty", ""))
        ref_modality_tokens = text_to_tokens(ref.get("Modality in Bucket", ""))
        ref_broad_tokens = text_to_tokens(ref.get("Broad Group", ""))

        specialty_score = overlap_score(paper_specialty_tokens, ref_specialty_tokens)
        modality_score = overlap_score(candidate_tokens, ref_modality_tokens)
        broad_score = overlap_score(candidate_tokens, ref_broad_tokens)

        total_score = (
            0.50 * name_score +
            0.25 * token_score +
            0.15 * specialty_score +
            0.05 * modality_score +
            0.05 * broad_score
        )

        if total_score > best_score:
            best_score = total_score
            best = {
                "matched_reference_dataset": ref_name,
                "matched_reference_sheet": ref.get("source_sheet", ""),
                "matched_reference_specialty": ref.get("Specialty", ""),
                "matched_reference_modality": ref.get("Modality in Bucket", ""),
                "matched_reference_broad_group": ref.get("Broad Group", ""),
                "name_score": round(name_score, 4),
                "token_score": round(token_score, 4),
                "specialty_score": round(specialty_score, 4),
                "modality_score": round(modality_score, 4),
                "broad_score": round(broad_score, 4),
                "usefulness_score": round(total_score, 4),
            }

    if not best:
        return {
            "usefulness_label": "DROP_NOT_USEFUL",
            "usefulness_reason": "No reference match found.",
            "usefulness_score": 0.0,
            "matched_reference_dataset": "",
            "matched_reference_sheet": "",
            "matched_reference_specialty": "",
            "matched_reference_modality": "",
            "matched_reference_broad_group": "",
            "name_score": 0.0,
            "token_score": 0.0,
            "specialty_score": 0.0,
            "modality_score": 0.0,
            "broad_score": 0.0,
        }

    # ===== Decision thresholds (UPDATED WITH SOFT RULE) =====
    if best["name_score"] >= 0.92 or best["usefulness_score"] >= 0.78:
        label = "KEEP_REFERENCE_MATCH"
        reason = "Strong match to reference dataset/topic catalog."

    elif best["specialty_score"] >= 0.9 and best["modality_score"] >= 0.9:
        label = "REVIEW_TOPIC_MATCH"
        reason = "Strong specialty + modality match, but weaker dataset name match."

    elif best["usefulness_score"] >= 0.52:
        label = "REVIEW_TOPIC_MATCH"
        reason = "Moderate topic match to reference catalog; review recommended."

    else:
        label = "DROP_NOT_USEFUL"
        reason = "Weak match to reference datasets/topics."

    return {
        **best,
        "usefulness_label": label,
        "usefulness_reason": reason,
    }

def is_real_dataset_source_url(url: str) -> bool:
    """
    Checks whether the URL itself looks like a real dataset source.
    This does not decide NimbleMinds usefulness yet.
    """
    if not url:
        return False

    try:
        parsed = urlparse(url)
        netloc = parsed.netloc.lower()
        path = parsed.path.lower()
        full = f"{netloc}{path}"

        # Strong dataset hosts / dataset pages
        if "zenodo.org" in netloc and "/records/" in path:
            return True

        if "kaggle.com" in netloc and "/datasets/" in path:
            return True

        if "huggingface.co" in netloc and "/datasets/" in path:
            return True

        dataset_domains = [
            "physionet.org",
            "figshare.com",
            "datadryad.org",
            "dryad.org",
            "osf.io",
            "openneuro.org",
            "cancerimagingarchive.net",
            "archive.ics.uci.edu",
            "openml.org",
            "brain-development.org",
            "ncbi.nlm.nih.gov",
            "bioproject.ncbi.nlm.nih.gov",
        ]

        if any(domain in netloc for domain in dataset_domains):
            return True

        # GitHub/GitLab only if clearly a data/dataset repository
        if "github.com" in netloc or "gitlab.com" in netloc:
            good_terms = ["dataset", "datasets", "data", "corpus", "benchmark"]
            bad_terms = [
                "model", "models", "tool", "tools", "framework",
                "train", "training", "inference", "code",
                "implementation", "demo", "library", "package"
            ]

            has_good = any(term in full for term in good_terms)
            has_bad = any(term in full for term in bad_terms)

            return has_good and not has_bad

        return False

    except Exception:
        return False


def save_dataset_metadata_csvs(rows: List[Dict[str, Any]], base_dir: str):
    """
    Saves metadata URL CSVs inside each specialty Metadata folder.

    Logic:
      1. Save ALL extracted article links into metadata_dataset_links_raw.csv.
      2. Score each link against the NimbleMinds reference catalog.
      3. Save only KEEP_REFERENCE_MATCH links into metadata_dataset_links_useful_only.csv.
      4. Aggregate across runs.
      5. Remove duplicates by arxiv_id + dataset_link.
    """
    metadata_rows_by_specialty = defaultdict(list)

    for row in rows:
        if not row.get("healthcare"):
            continue

        specialty = row.get("specialty", "Other")
        specialty_folder = _slugify(specialty)

        all_links_text = row.get("dataset_links", "")
        dataset_mentions_text = row.get("dataset_mentions", "")

        if not all_links_text:
            continue

        links = [link.strip() for link in all_links_text.split("|") if link.strip()]
        mentions = [m.strip() for m in dataset_mentions_text.split(" | ") if m.strip()]

        for link in links:
            # Score every link, but only useful ones will go to useful_only.
            usefulness = classify_dataset_usefulness(
                row=row,
                dataset_link=link,
                dataset_mentions=mentions,
            )

            metadata_row = {
                "arxiv_id": row.get("arxiv_id", ""),
                "title": row.get("title", ""),
                "specialty": specialty,
                "pdf_url": row.get("pdf_url", ""),
                "dataset_link": link,
                "dataset_mentions": dataset_mentions_text,
                "is_dataset_like_url": is_dataset_like_url(link),
                **usefulness,
            }

            metadata_rows_by_specialty[specialty_folder].append(metadata_row)

    # Include folders that already have metadata files,
    # even if this run found no new links.
    specialty_folders = set(metadata_rows_by_specialty.keys())

    for folder_name in ["radiology", "oncology", "pathology", "other"]:
        metadata_dir = os.path.join(base_dir, folder_name, "Metadata")
        raw_path = os.path.join(metadata_dir, "metadata_dataset_links_raw.csv")

        if os.path.exists(raw_path):
            specialty_folders.add(folder_name)

    default_fieldnames = [
        "arxiv_id",
        "title",
        "specialty",
        "pdf_url",
        "dataset_link",
        "dataset_mentions",
        "is_dataset_like_url",
        "matched_reference_dataset",
        "matched_reference_sheet",
        "matched_reference_specialty",
        "matched_reference_modality",
        "matched_reference_broad_group",
        "name_score",
        "token_score",
        "specialty_score",
        "modality_score",
        "broad_score",
        "usefulness_score",
        "usefulness_label",
        "usefulness_reason",
    ]

    for specialty_folder in sorted(specialty_folders):
        new_rows = metadata_rows_by_specialty.get(specialty_folder, [])

        metadata_dir = os.path.join(base_dir, specialty_folder, "Metadata")
        os.makedirs(metadata_dir, exist_ok=True)

        raw_path = os.path.join(metadata_dir, "metadata_dataset_links_raw.csv")
        all_path = os.path.join(metadata_dir, "metadata_dataset_links.csv")
        useful_path = os.path.join(metadata_dir, "metadata_dataset_links_useful_only.csv")

        old_rows = []
        if os.path.exists(raw_path):
            try:
                old_df = pd.read_csv(raw_path).fillna("")
                old_rows = old_df.to_dict("records")
            except Exception as e:
                print(f"[Datasets] Could not read old metadata CSV for {specialty_folder}: {e}")

        combined_rows = old_rows + new_rows

        # De-duplicate by arxiv_id + dataset_link
        deduped = {}
        for r in combined_rows:
            key = (
                str(r.get("arxiv_id", "")).strip(),
                str(r.get("dataset_link", "")).strip(),
            )
            if key[0] and key[1]:
                deduped[key] = r

        combined_rows = list(deduped.values())

        if combined_rows:
            fieldnames = sorted({k for r in combined_rows for k in r.keys()})
        else:
            fieldnames = default_fieldnames

        # RAW = all links from the article
        save_rows_to_csv_allow_empty(combined_rows, raw_path, fieldnames)

        # Main dataset links CSV can also hold all links with scores
        save_rows_to_csv_allow_empty(combined_rows, all_path, fieldnames)

        # USEFUL ONLY = only links that score as useful to NimbleMinds
        useful_rows = [
            r for r in combined_rows
            if (
                r.get("usefulness_label") in {"KEEP_REFERENCE_MATCH", "REVIEW_TOPIC_MATCH"}
                or str(r.get("is_dataset_like_url", "")).lower() == "true"
                or float(r.get("usefulness_score", 0) or 0) >= 0.35
            )
        ]

        save_rows_to_csv_allow_empty(useful_rows, useful_path, fieldnames)

        print(
            f"[Datasets] Saved aggregated metadata CSVs for {specialty_folder}: "
            f"old={len(old_rows)}, new={len(new_rows)}, "
            f"combined={len(combined_rows)}, useful={len(useful_rows)}"
        )

In [ ]:
#fixing previous dataset bug! no need to run if you will run the agent
save_dataset_metadata_csvs([], CLASSIFIED_DIR)

[Save] Wrote 58 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/oncology/Metadata/metadata_dataset_links_raw.csv
[Save] Wrote 58 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/oncology/Metadata/metadata_dataset_links.csv
[Save] Wrote 46 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/oncology/Metadata/metadata_dataset_links_useful_only.csv
[Datasets] Saved aggregated metadata CSVs for oncology: old=58, new=0, combined=58, useful=46
[Save] Wrote 394 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/Metadata/metadata_dataset_links_raw.csv
[Save] Wrote 394 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/Metadata/metadata_dataset_links.csv
[Save] Wrote 50 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/Metadata/metadata_dataset_links_useful_only.csv
[Datasets] Saved aggregated metadata CSVs for other: 

# Testing - Image Classifier

In [ ]:
import pandas as pd
import re

REFERENCE_IMAGE_XLSX_PATH = "/content/drive/MyDrive/AI/2025 Team/Datasets/Copy of Proprietary Training Data Summary - Running Total v2.xlsx"


def normalize_text(text):
    return re.sub(r"\s+", " ", str(text or "").lower()).strip()


def load_reference_image_terms(xlsx_path):
    """
    Loads the proprietary training data summary and extracts image-related reference terms.
    This becomes the target profile for image filtering.
    """
    sheets = pd.read_excel(xlsx_path, sheet_name=None)
    all_text = []

    for sheet_name, df in sheets.items():
        df = df.fillna("")
        for _, row in df.iterrows():
            row_text = " ".join(str(v) for v in row.values)
            all_text.append(row_text)

    combined_text = normalize_text(" ".join(all_text))

    reference_terms = []

    candidate_terms = [
        "ct", "computed tomography", "mri", "magnetic resonance",
        "x-ray", "xray", "radiograph", "ultrasound", "sonography",
        "histology", "histopathology", "pathology", "microscopy",
        "microscopic", "retina", "retinal", "fundus",
        "endoscopy", "dermoscopy", "dermatology",
        "segmentation", "lesion", "tumor", "tumour",
        "brain", "chest", "lung", "heart", "cardiac",
        "medical image", "radiology image", "scan"
    ]

    for term in candidate_terms:
        if term in combined_text:
            reference_terms.append(term)

    reference_terms = sorted(list(set(reference_terms)))

    print(f"[Reference Images] Loaded {len(reference_terms)} reference terms:")
    print(reference_terms)

    return reference_terms


REFERENCE_IMAGE_TERMS = load_reference_image_terms(REFERENCE_IMAGE_XLSX_PATH)

[Reference Images] Loaded 22 reference terms:
['brain', 'chest', 'ct', 'dermatology', 'dermoscopy', 'endoscopy', 'fundus', 'heart', 'histopathology', 'lesion', 'lung', 'microscopy', 'mri', 'pathology', 'retina', 'retinal', 'scan', 'segmentation', 'tumor', 'ultrasound', 'x-ray', 'xray']


In [ ]:
import os
import re
import csv
import math
import shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False


# ---------------------------
# HELPERS
# ---------------------------

def _safe_ratio(a, b):
    return float(a) / float(b) if b else 0.0


def _load_image_rgb(image_path: str):
    return Image.open(image_path).convert("RGB")


def _grayscale_fraction(arr: np.ndarray) -> float:
    r = arr[:, :, 0].astype(np.int16)
    g = arr[:, :, 1].astype(np.int16)
    b = arr[:, :, 2].astype(np.int16)
    close = (np.abs(r - g) < 8) & (np.abs(r - b) < 8) & (np.abs(g - b) < 8)
    return float(close.mean())


def _white_fraction(arr: np.ndarray) -> float:
    return float(((arr[:, :, 0] > 245) & (arr[:, :, 1] > 245) & (arr[:, :, 2] > 245)).mean())


def _black_fraction(arr: np.ndarray) -> float:
    return float(((arr[:, :, 0] < 15) & (arr[:, :, 1] < 15) & (arr[:, :, 2] < 15)).mean())


def _edge_density(gray_u8: np.ndarray) -> float:
    if CV2_AVAILABLE:
        edges = cv2.Canny(gray_u8, 50, 150)
        return float((edges > 0).mean())

    pil_gray = Image.fromarray(gray_u8)
    edges = pil_gray.filter(ImageFilter.FIND_EDGES)
    arr = np.array(edges)
    return float((arr > 25).mean())


def _line_density(gray_u8: np.ndarray) -> float:
    if not CV2_AVAILABLE:
        return 0.0

    lines = cv2.HoughLinesP(
        gray_u8,
        rho=1,
        theta=np.pi / 180,
        threshold=60,
        minLineLength=max(20, gray_u8.shape[1] // 8),
        maxLineGap=6,
    )

    if lines is None:
        return 0.0

    return min(len(lines) / 150.0, 1.0)


def _entropy(gray_u8: np.ndarray) -> float:
    hist, _ = np.histogram(gray_u8.flatten(), bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]

    if len(hist) == 0:
        return 0.0

    return float(-(hist * np.log2(hist)).sum())


def _center_vs_border_signal(gray_u8: np.ndarray) -> float:
    h, w = gray_u8.shape

    if h < 20 or w < 20:
        return 0.0

    y1, y2 = h // 4, 3 * h // 4
    x1, x2 = w // 4, 3 * w // 4
    center = gray_u8[y1:y2, x1:x2]

    border_mask = np.ones_like(gray_u8, dtype=bool)
    border_mask[y1:y2, x1:x2] = False
    border = gray_u8[border_mask]

    center_std = float(center.std())
    border_std = float(border.std()) if border.size else 0.0

    return center_std - border_std


def _colorfulness(arr: np.ndarray) -> float:
    arr = arr.astype(np.float32)

    rg = np.abs(arr[:, :, 0] - arr[:, :, 1])
    yb = np.abs(0.5 * (arr[:, :, 0] + arr[:, :, 1]) - arr[:, :, 2])

    std_rg, std_yb = np.std(rg), np.std(yb)
    mean_rg, mean_yb = np.mean(rg), np.mean(yb)

    return float(np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2))


def _normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").lower()).strip()


# ---------------------------
# KEYWORD SIGNALS
# ---------------------------

POSITIVE_KEYWORDS = [
    "ct", "computed tomography", "mri", "magnetic resonance",
    "x-ray", "xray", "radiograph", "radiology", "ultrasound",
    "sonograph", "pet", "spect", "microscopy", "microscopic",
    "histology", "histopathology", "pathology", "endoscopy",
    "dermoscopy", "fundus", "retina", "retinal", "lesion",
    "tumor", "tumour", "hemorrhage", "haemorrhage", "brain scan",
    "scan", "segmentation", "medical image", "patient image"
]

NEGATIVE_KEYWORDS = [
    "pipeline", "workflow", "framework", "architecture", "overview",
    "diagram", "schematic", "flowchart", "algorithm", "chart", "plot",
    "graph", "table", "bar chart", "line chart", "box plot", "study design",
    "experimental setup", "method overview", "block diagram"
]


def caption_keyword_score(caption: str):
    caption_norm = _normalize_text(caption)

    pos_hits = [kw for kw in POSITIVE_KEYWORDS if kw in caption_norm]
    neg_hits = [kw for kw in NEGATIVE_KEYWORDS if kw in caption_norm]

    score = 0.0
    score += min(len(pos_hits) * 0.9, 2.7)
    score -= min(len(neg_hits) * 1.0, 3.0)

    return score, pos_hits, neg_hits


# ---------------------------
# PROPRIETARY REFERENCE MATCHING
# ---------------------------

def reference_image_match_score(caption: str, reference_terms=None):
    """
    Checks whether the extracted image caption matches the proprietary
    training-data image profile loaded from the reference spreadsheet.
    """
    if reference_terms is None:
        reference_terms = globals().get("REFERENCE_IMAGE_TERMS", [])

    caption_norm = _normalize_text(caption)

    if not reference_terms:
        return 0.0, []

    hits = []
    for term in reference_terms:
        term_norm = _normalize_text(term)
        if term_norm and term_norm in caption_norm:
            hits.append(term)

    score = min(len(hits) * 1.2, 4.0)

    return score, hits


# ---------------------------
# MAIN IMAGE CLASSIFIER
# ---------------------------

def classify_image_usefulness(image_path: str, caption: str = "", figure_label: str = "") -> dict:
    """
    Returns:
      KEEP_MEDICAL / REVIEW_BORDERLINE / DROP_NON_MEDICAL

    Uses:
      1. visual image features
      2. caption keyword evidence
      3. proprietary reference-term similarity
    """

    img = _load_image_rgb(image_path)
    arr = np.array(img)
    gray = np.array(ImageOps.grayscale(img))

    h, w = gray.shape
    aspect_ratio = _safe_ratio(w, h)
    megapixels = (w * h) / 1_000_000

    grayscale_frac = _grayscale_fraction(arr)
    white_frac = _white_fraction(arr)
    black_frac = _black_fraction(arr)
    edge_density = _edge_density(gray)
    line_density = _line_density(gray)
    entropy = _entropy(gray)
    center_signal = _center_vs_border_signal(gray)
    colorfulness = _colorfulness(arr)

    medical_score = 0.0
    diagram_score = 0.0
    reasons = []

    # ---- visual positives ----
    if grayscale_frac > 0.80:
        medical_score += 1.4
        reasons.append("high grayscale content")

    if black_frac > 0.35 and center_signal > 8:
        medical_score += 1.4
        reasons.append("dark background with central signal")

    if 4.2 <= entropy <= 7.6:
        medical_score += 1.0
        reasons.append("organic texture")

    if 0.03 <= edge_density <= 0.22:
        medical_score += 0.9
        reasons.append("moderate structural detail")

    if grayscale_frac < 0.55 and colorfulness > 18 and entropy > 5.0:
        medical_score += 1.0
        reasons.append("photo/tissue-like color texture")

    if megapixels >= 0.08:
        medical_score += 0.4

    # ---- visual negatives ----
    if white_frac > 0.70:
        diagram_score += 1.2
        reasons.append("mostly white background")

    if line_density > 0.22:
        diagram_score += 1.8
        reasons.append("strong straight-line geometry")

    if edge_density < 0.015:
        diagram_score += 0.8
        reasons.append("too little image structure")

    if entropy < 3.2:
        diagram_score += 1.4
        reasons.append("low texture / flat regions")

    if aspect_ratio > 2.8 or aspect_ratio < 0.35:
        diagram_score += 0.7
        reasons.append("extreme aspect ratio")

    if w < 180 or h < 180:
        diagram_score += 1.5
        reasons.append("too small to be useful")

    # ---- caption text signal ----
    text_score, pos_hits, neg_hits = caption_keyword_score(caption)

    if text_score > 0:
        medical_score += text_score
        reasons.append(f"caption medical keywords: {', '.join(pos_hits[:4])}")
    elif text_score < 0:
        diagram_score += abs(text_score)
        reasons.append(f"caption non-medical keywords: {', '.join(neg_hits[:4])}")

    # ---- proprietary reference signal ----
    reference_score, reference_hits = reference_image_match_score(caption)

    if reference_score > 0:
        medical_score += reference_score
        reasons.append(f"matches proprietary reference terms: {', '.join(reference_hits[:5])}")
    else:
        diagram_score += 0.5
        reasons.append("no match to proprietary reference image profile")

    # slight boost if it is explicitly a figure caption
    if figure_label:
        medical_score += 0.15

    final_score = medical_score - diagram_score

    if final_score >= 1.25:
        label = "KEEP_MEDICAL"
    elif final_score <= -0.75:
        label = "DROP_NON_MEDICAL"
    else:
        label = "REVIEW_BORDERLINE"

    confidence = min(abs(final_score) / 3.0, 1.0)

    return {
        "label": label,
        "is_medical": label == "KEEP_MEDICAL",
        "confidence": round(confidence, 3),
        "reason": (
            f"medical_score={medical_score:.2f}, "
            f"diagram_score={diagram_score:.2f}, "
            f"reference_score={reference_score:.2f}, "
            f"final_score={final_score:.2f}"
        ),
        "features": {
            "width": w,
            "height": h,
            "aspect_ratio": round(aspect_ratio, 3),
            "megapixels": round(megapixels, 3),
            "grayscale_fraction": round(grayscale_frac, 3),
            "white_fraction": round(white_frac, 3),
            "black_fraction": round(black_frac, 3),
            "edge_density": round(edge_density, 3),
            "line_density": round(line_density, 3),
            "entropy": round(entropy, 3),
            "center_signal": round(center_signal, 3),
            "colorfulness": round(colorfulness, 3),
            "medical_score": round(medical_score, 3),
            "diagram_score": round(diagram_score, 3),
            "reference_score": round(reference_score, 3),
            "final_score": round(final_score, 3),
            "positive_caption_hits": " | ".join(pos_hits),
            "negative_caption_hits": " | ".join(neg_hits),
            "reference_hits": " | ".join(reference_hits),
            "trigger_notes": " | ".join(reasons[:10]),
            "caption": caption,
            "figure_label": figure_label,
        },
    }


# ---------------------------
# FILTER ONE PAPER FOLDER
# ---------------------------

def filter_images_in_paper_folder(
    paper_image_dir: str,
    save_csv: bool = True,
    move_files: bool = False,
):
    """
    Reads image_metadata.csv if present, classifies each image, and optionally
    moves files into _keep / _review / _drop folders.
    """
    metadata_csv = os.path.join(paper_image_dir, "image_metadata.csv")
    rows = []

    if os.path.exists(metadata_csv):
        df_meta = pd.read_csv(metadata_csv).fillna("")
        rows = df_meta.to_dict("records")
    else:
        rows = []
        for f in os.listdir(paper_image_dir):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                rows.append({
                    "saved_filename": f,
                    "saved_path": os.path.join(paper_image_dir, f),
                    "caption": "",
                    "figure_label": "",
                })

    results = []

    keep_dir = os.path.join(paper_image_dir, "_keep")
    review_dir = os.path.join(paper_image_dir, "_review")
    drop_dir = os.path.join(paper_image_dir, "_drop")

    if move_files:
        os.makedirs(keep_dir, exist_ok=True)
        os.makedirs(review_dir, exist_ok=True)
        os.makedirs(drop_dir, exist_ok=True)

    for row in rows:
        image_path = row.get("saved_path", "")

        if not image_path or not os.path.exists(image_path):
            image_path = os.path.join(paper_image_dir, row.get("saved_filename", ""))

        if not os.path.exists(image_path):
            continue

        caption = row.get("caption", "")
        figure_label = row.get("figure_label", "")

        try:
            analysis = classify_image_usefulness(
                image_path=image_path,
                caption=caption,
                figure_label=figure_label,
            )

            result_row = {
                "saved_filename": os.path.basename(image_path),
                "saved_path": image_path,
                "label": analysis["label"],
                "confidence": analysis["confidence"],
                "reason": analysis["reason"],
                "caption": caption,
                "figure_label": figure_label,
            }

            result_row.update(analysis["features"])
            results.append(result_row)

            if move_files:
                dst_dir = {
                    "KEEP_MEDICAL": keep_dir,
                    "REVIEW_BORDERLINE": review_dir,
                    "DROP_NON_MEDICAL": drop_dir,
                }[analysis["label"]]

                dst_path = os.path.join(dst_dir, os.path.basename(image_path))

                if os.path.abspath(image_path) != os.path.abspath(dst_path):
                    shutil.copy2(image_path, dst_path)

        except Exception as e:
            print(f"[Image Filter] Failed on {image_path}: {e}")

    if save_csv and results:
        output_csv = os.path.join(paper_image_dir, "image_filter_results.csv")
        pd.DataFrame(results).to_csv(output_csv, index=False)

    summary = {
        "total": len(results),
        "keep_medical": sum(r["label"] == "KEEP_MEDICAL" for r in results),
        "review_borderline": sum(r["label"] == "REVIEW_BORDERLINE" for r in results),
        "drop_non_medical": sum(r["label"] == "DROP_NON_MEDICAL" for r in results),
    }

    return results, summary


# ---------------------------
# FILTER ALL PAPERS UNDER CLASSIFIED_DIR
# ---------------------------

def generate_image_filter_report(classified_dir: str, output_csv: str = None):
    all_rows = []

    for specialty in ["radiology", "oncology", "pathology", "other"]:
        specialty_dir = os.path.join(classified_dir, specialty)
        images_root = os.path.join(specialty_dir, "Metadata", "images")

        if not os.path.isdir(images_root):
            continue

        for paper_folder in os.listdir(images_root):
            paper_image_dir = os.path.join(images_root, paper_folder)

            if not os.path.isdir(paper_image_dir):
                continue

            try:
                rows, summary = filter_images_in_paper_folder(
                    paper_image_dir=paper_image_dir,
                    save_csv=True,
                    move_files=False,
                )

                for r in rows:
                    r["specialty"] = specialty
                    r["paper_folder"] = paper_folder
                    all_rows.append(r)

                print(
                    f"[Image Filter] {specialty} / {paper_folder}: "
                    f"total={summary['total']} | "
                    f"keep={summary['keep_medical']} | "
                    f"review={summary['review_borderline']} | "
                    f"drop={summary['drop_non_medical']}"
                )

            except Exception as e:
                print(f"[Image Filter] Failed on {paper_image_dir}: {e}")

    if output_csv is None:
        output_csv = os.path.join(classified_dir, "image_filter_report.csv")

    if all_rows:
        pd.DataFrame(all_rows).to_csv(output_csv, index=False)
        print(f"\n[Image Filter] Report saved -> {output_csv}")
    else:
        print("\n[Image Filter] No rows to save.")

    return all_rows

In [ ]:
# ---------------------------
# QUICK IMAGE FILTER TEST
# ---------------------------

import os
import pandas as pd

print("REFERENCE_IMAGE_TERMS loaded:", "REFERENCE_IMAGE_TERMS" in globals())
if "REFERENCE_IMAGE_TERMS" in globals():
    print("Number of reference terms:", len(REFERENCE_IMAGE_TERMS))
    print("Sample reference terms:", REFERENCE_IMAGE_TERMS[:20])

print("\nCLASSIFIED_DIR:", CLASSIFIED_DIR)

# Run filter report across all extracted image folders
report_rows = generate_image_filter_report(CLASSIFIED_DIR)

print("\nTotal images classified:", len(report_rows))

if report_rows:
    df_test = pd.DataFrame(report_rows)

    print("\nLabel counts:")
    print(df_test["label"].value_counts())

    display_cols = [
        "specialty",
        "paper_folder",
        "saved_filename",
        "label",
        "confidence",
        "reference_score",
        "reference_hits",
        "positive_caption_hits",
        "negative_caption_hits",
        "caption",
        "trigger_notes",
    ]

    display_cols = [c for c in display_cols if c in df_test.columns]

    display(df_test[display_cols].head(30))

    output_path = os.path.join(CLASSIFIED_DIR, "image_filter_test_preview.csv")
    df_test.to_csv(output_path, index=False)
    print("\nSaved test preview to:", output_path)
else:
    print("\nNo images were classified.")
    print("Possible reasons:")
    print("1. Images were not extracted yet.")
    print("2. Image folders are empty.")
    print("3. CLASSIFIED_DIR path is wrong.")

REFERENCE_IMAGE_TERMS loaded: True
Number of reference terms: 22
Sample reference terms: ['brain', 'chest', 'ct', 'dermatology', 'dermoscopy', 'endoscopy', 'fundus', 'heart', 'histopathology', 'lesion', 'lung', 'microscopy', 'mri', 'pathology', 'retina', 'retinal', 'scan', 'segmentation', 'tumor', 'ultrasound']

CLASSIFIED_DIR: /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified
[Image Filter] radiology / 2303.00111v2_PixCUE Joint Uncertainty Estimation and Image Reconstruction in MRI using Deep Pixel Classification: total=7 | keep=3 | review=4 | drop=0
[Image Filter] radiology / 2302.14808v2_Opto-UNet Optimized UNet for Segmentation of Varicose Veins in Optical Coherence Tomography: total=4 | keep=2 | review=1 | drop=1
[Image Filter] radiology / 2302.14795v1_3D Coronary Vessel Reconstruction from Bi-Plane Angiography using Graph Convolutional Networks: total=4 | keep=1 | review=0 | drop=3
[Image Filter] radiology / 2302.14490v1_Estimating Head Motion from MR-Images: t

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (102947328 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (150874752 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[Image Filter] radiology / 2302.13390v3_MDF-Net for abnormality detection by fusing X-rays with clinical data: total=7 | keep=2 | review=3 | drop=2
[Image Filter] radiology / 2302.13336v2_Key-Exchange Convolutional Auto-Encoder for Data Augmentation in Early Knee Osteoarthritis Detection: total=9 | keep=2 | review=3 | drop=4
[Image Filter] radiology / 2302.13251v2_Unsupervised Domain Adaptation for Low-dose CT Reconstruction via Bayesian Uncertainty Alignment: total=87 | keep=61 | review=26 | drop=0
[Image Filter] radiology / 2302.13207v1_Stereo X-ray Tomography: total=20 | keep=7 | review=11 | drop=2
[Image Filter] radiology / 2302.13195v1_nnUNet RASPP for Retinal OCT Fluid Detection, Segmentation and Generalisation over Variations of Data Sourc: total=14 | keep=5 | review=5 | drop=4
[Image Filter] radiology / 2302.13172v1_Deep Learning-based Multi-Organ CT Segmentation with Adversarial Data Augmentation: total=3 | keep=3 | review=0 | drop=0
[Image Filter] oncology / 2302.14124v1_Mult

,specialty,paper_folder,saved_filename,label,confidence,reference_score,reference_hits,positive_caption_hits,negative_caption_hits,caption,trigger_notes
0,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_7_img_1_where_is_the_normalized_pixcue_un...,KEEP_MEDICAL,0.533,1.2,ct,ct,,"where 𝑈𝑝,𝑟 is the normalized PixCUE uncertaint...",high grayscale content | moderate structural d...
1,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_11_img_1.jpg,KEEP_MEDICAL,0.467,0.0,,,,,dark background with central signal | organic ...
2,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_11_img_2_fig_2.jpg,KEEP_MEDICAL,0.883,1.2,ct,ct | spect,framework,Fig. 2: Results for Experiment I. Input: zero-...,organic texture | moderate structural detail |...
3,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_12_img_1.jpg,REVIEW_BORDERLINE,0.000,0.0,,,,,organic texture | moderate structural detail |...
4,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_12_img_2_fig_4.jpg,REVIEW_BORDERLINE,0.250,1.2,ct,ct,framework | algorithm,Fig. 4: Results for Experiment III. input: zer...,organic texture | moderate structural detail |...
5,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_13_img_1.jpg,REVIEW_BORDERLINE,0.133,0.0,,,,,high grayscale content | moderate structural d...
6,radiology,2303.00111v2_PixCUE Joint Uncertainty Estimati...,page_13_img_2_fig_6.png,REVIEW_BORDERLINE,0.150,1.2,ct,ct,framework,Fig. 6: Results for Experiment V. (a) PixCUE: ...,high grayscale content | moderate structural d...
7,radiology,2302.14808v2_Opto-UNet Optimized UNet for Segm...,page_4_img_1_fig1.png,DROP_NON_MEDICAL,0.950,1.2,segmentation,segmentation,diagram | block diagram,Fig.1 Block diagram of the proposed model Opto...,moderate structural detail | mostly white back...
8,radiology,2302.14808v2_Opto-UNet Optimized UNet for Segm...,page_5_img_1_fig_3.jpg,KEEP_MEDICAL,1.000,1.2,ct,ct,,Fig. 3 (a) Original Varicose vein OCT image (b...,high grayscale content | moderate structural d...
9,radiology,2302.14808v2_Opto-UNet Optimized UNet for Segm...,page_5_img_2_fig_2.jpg,KEEP_MEDICAL,1.000,2.4,ct | segmentation,ct | segmentation,,Fig. 2 (a) Original Varicose vein OCT image (b...,high grayscale content | organic texture | mod...



Saved test preview to: /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/image_filter_test_preview.csv


# Testing - Dataset Classifier *DONE*

In [ ]:
print("Reference catalog rows:", len(REFERENCE_CATALOG))
print(REFERENCE_CATALOG[[
    "File Name",
    "Broad Group",
    "Specialty",
    "Modality in Bucket",
    "source_sheet"
]].head(10))

Reference catalog rows: 178
                      File Name Broad Group         Specialty  \
0                 PolypsSet.zip   2D Images  Gastroenterology   
1     Breast_Histopathology.zip   2D Images         Pathology   
2                      SNOW.zip   2D Images         Pathology   
3           Skin_DNA_Damage.zip   2D Images         Pathology   
4  DeepLesion_DeepLesion_01.zip    CT scans         Radiology   
5         colorectal_cancer.zip   2D Images         Pathology   
6                      CXR8.zip   2D Images         Radiology   
7              PatchGastric.zip   2D Images         Pathology   
8                  Path_VQA.zip   2D Images         Pathology   
9                    PMC_OA.zip   2D Images         Radiology   

  Modality in Bucket source_sheet  
0            Imaging    Data List  
1            Imaging    Data List  
2            Imaging    Data List  
3            Imaging    Data List  
4            Imaging    Data List  
5            Imaging    Data List  
6   

In [ ]:
test_row = {
    "arxiv_id": "test-001",
    "title": "A study using public oncology imaging datasets for tumor classification",
    "abstract": "We used a public cancer imaging dataset and benchmark data for model development.",
    "specialty": "Oncology",
    "pdf_url": "https://example.com/test.pdf",
}

test_dataset_link = "https://openneuro.org/datasets/ds003097/versions/1.2.1"
test_dataset_mentions = [
    "We evaluated the method on a public dataset for tumor classification.",
    "The dataset is openly available and commonly used for benchmarking."
]

result = classify_dataset_usefulness(
    row=test_row,
    dataset_link=test_dataset_link,
    dataset_mentions=test_dataset_mentions,
)

print(result)

{'matched_reference_dataset': 'LUNA16.zip', 'matched_reference_sheet': 'Data List', 'matched_reference_specialty': 'Pulmonology / Oncology', 'matched_reference_modality': 'Imaging', 'matched_reference_broad_group': '3D Imaging (CT)', 'name_score': 0.1818, 'token_score': 0.5, 'specialty_score': 1.0, 'modality_score': 1.0, 'broad_score': 0.5, 'usefulness_score': 0.4409, 'usefulness_label': 'REVIEW_TOPIC_MATCH', 'usefulness_reason': 'Strong specialty + modality match, but weaker dataset name match.'}


In [ ]:
for k, v in result.items():
    print(f"{k}: {v}")

matched_reference_dataset: LUNA16.zip
matched_reference_sheet: Data List
matched_reference_specialty: Pulmonology / Oncology
matched_reference_modality: Imaging
matched_reference_broad_group: 3D Imaging (CT)
name_score: 0.1818
token_score: 0.5
specialty_score: 1.0
modality_score: 1.0
broad_score: 0.5
usefulness_score: 0.4409
usefulness_label: REVIEW_TOPIC_MATCH
usefulness_reason: Strong specialty + modality match, but weaker dataset name match.


In [ ]:
test_pdf_url = "https://arxiv.org/pdf/2301.00785v5"

dataset_info = extract_dataset_info_from_pdf_url(test_pdf_url)

test_row_real = {
    "arxiv_id": "2301.03459v1",
    "title": "test paper",
    "abstract": "",
    "specialty": "Oncology",
    "pdf_url": test_pdf_url,
}

if dataset_info["dataset_links"]:
    for link in dataset_info["dataset_links"]:
        result = classify_dataset_usefulness(
            row=test_row_real,
            dataset_link=link,
            dataset_mentions=dataset_info["dataset_mentions"],
        )
        print("\nLINK:", link)
        for k, v in result.items():
            print(f"{k}: {v}")
else:
    result = classify_dataset_usefulness(
        row=test_row_real,
        dataset_link="",
        dataset_mentions=dataset_info["dataset_mentions"],
    )
    print("\nNO DATASET LINK FOUND")
    for k, v in result.items():
        print(f"{k}: {v}")


LINK: https://github.com/ljwztc/CLIP-Driven-Universal-Model
matched_reference_dataset: LIDC-IDRI_CT.zip
matched_reference_sheet: Data List
matched_reference_specialty: Pulmonology / Oncology
matched_reference_modality: Imaging
matched_reference_broad_group: 3D Imaging
name_score: 0.2564
token_score: 0.4
specialty_score: 1.0
modality_score: 1.0
broad_score: 1.0
usefulness_score: 0.4782
usefulness_label: REVIEW_TOPIC_MATCH
usefulness_reason: Strong specialty + modality match, but weaker dataset name match.

LINK: https://monai.io/
matched_reference_dataset: LIDC-IDRI_CT.zip
matched_reference_sheet: Data List
matched_reference_specialty: Pulmonology / Oncology
matched_reference_modality: Imaging
matched_reference_broad_group: 3D Imaging
name_score: 0.0136
token_score: 0.4
specialty_score: 1.0
modality_score: 1.0
broad_score: 1.0
usefulness_score: 0.3568
usefulness_label: REVIEW_TOPIC_MATCH
usefulness_reason: Strong specialty + modality match, but weaker dataset name match.

LINK: https:/

In [ ]:
#def preview_reference_catalog(ref_df: pd.DataFrame, n: int = 10) -> None:
#    print("Reference catalog rows:", len(ref_df))
#    print(ref_df[[
#        "File Name",
#        "Broad Group",
#        "Specialty",
#        "Modality in Bucket",
#        "source_sheet"
#    ]].head(n))

#4. Agent

In [ ]:
def run_pipeline(
    start_date: str,
    end_date: str,
    categories=None,
    max_results_total: int = 200,
    arxiv_start_offset: int = 0,
    run_label: str = "",
):
    print("=" * 60)
    print("Agent started")
    print(f"Date range: {start_date} to {end_date}")
    output_start = start_date
    output_end = end_date

    if run_label:
        output_start = f"{start_date}_{run_label}"

   # if show_reference_preview:
   #     preview_reference_catalog(REFERENCE_CATALOG)
    # Step 1: Fetch papers
    papers = fetch_papers_by_date_range(
        start_date=start_date,
        end_date=end_date,
        categories=categories or DEFAULT_CATEGORIES,
        max_results_total=max_results_total,
        page_size=25,
        chunk_days=31,
        timeout=60,
        polite_delay_seconds=15.0,
        start_offset=arxiv_start_offset,
    )

    print(f"Fetched {len(papers)} papers")

    # Optional: save fetched papers
    fetched_path = os.path.join(FETCHED_DIR, f"{output_start}_{output_end}_fetched.csv")
    save_rows_to_csv(papers, fetched_path)

    # Step 2: Classify
    rows, summary = classify_batch(papers)

    print("Classification summary:")
    for k, v in summary.items():
        print(f"  {k}: {v}")

    # Step 3: Extract full text + abstract
    save_full_texts_by_specialty(
        rows,
        base_dir=CLASSIFIED_DIR,
    )

    # Step 4: Extract dataset links + mentions
    rows = add_dataset_info_to_rows(rows)

    # Step 5: Save dataset metadata CSVs
    save_dataset_metadata_csvs(
        rows,
        base_dir=CLASSIFIED_DIR,
    )

    # Step 6: Save extracted images
    save_image_assets(
        rows,
        base_dir=CLASSIFIED_DIR,
    )
    # Step 7: Filter extracted images
    generate_image_filter_report(CLASSIFIED_DIR)

    # Step 8: Save classified CSV outputs
    save_specialty_outputs(
        rows,
        base_dir=CLASSIFIED_DIR,
        start_date=output_start,
        end_date=end_date,
    )

    # Step 9: Save summary
    summary_path = os.path.join(CLASSIFIED_DIR, f"{output_start}_{output_end}_summary.csv")
    save_run_summary(summary, summary_path)

    print("Pipeline complete")
    print("=" * 60)

    return papers, rows, summary

In [ ]:
papers, rows, summary = run_pipeline(
    start_date="2023-02-01",
    end_date="2023-02-28",
    max_results_total=100,
    arxiv_start_offset=900,
    run_label="batch_010",
)

Agent started
Date range: 2023-02-01 to 2023-02-28
[Fetcher] 2023-02-01 → 2023-02-28 | pulled so far: 100
[Fetcher] DONE | Total unique papers: 100
Fetched 100 papers
[Save] Wrote 100 rows -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Fetched/2023-02-01_batch_010_2023-02-28_fetched.csv (exists=True, size=163072)


Classifying papers:   0%|          | 0/100 [00:00<?, ?paper/s]

Classification summary:
  total: 100
  healthcare: 8
  healthcare_rate: 0.08
  specialties: {'Radiology': 3, 'Oncology': 0, 'Pathology': 1, 'Other': 4}
[Save] Appended article text -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/other_full_articles.txt
[Save] Appended article text -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/radiology/radiology_full_articles.txt
[Save] Appended article text -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/other_full_articles.txt
[Save] Appended article text -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/other_full_articles.txt
[Save] Appended article text -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/other/other_full_articles.txt
[Save] Appended article text -> /content/drive/MyDrive/AI/2025 Team/Scraping Agent/Output/Classified/pathology/pathology_full_articles.txt
[Save] Appended article text -> /content/drive

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (102947328 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (150874752 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[Image Filter] radiology / 2302.13390v3_MDF-Net for abnormality detection by fusing X-rays with clinical data: total=7 | keep=2 | review=3 | drop=2
[Image Filter] radiology / 2302.13336v2_Key-Exchange Convolutional Auto-Encoder for Data Augmentation in Early Knee Osteoarthritis Detection: total=9 | keep=2 | review=3 | drop=4
[Image Filter] radiology / 2302.13251v2_Unsupervised Domain Adaptation for Low-dose CT Reconstruction via Bayesian Uncertainty Alignment: total=87 | keep=61 | review=26 | drop=0
[Image Filter] radiology / 2302.13207v1_Stereo X-ray Tomography: total=20 | keep=7 | review=11 | drop=2
[Image Filter] radiology / 2302.13195v1_nnUNet RASPP for Retinal OCT Fluid Detection, Segmentation and Generalisation over Variations of Data Sourc: total=14 | keep=5 | review=5 | drop=4
[Image Filter] radiology / 2302.13172v1_Deep Learning-based Multi-Organ CT Segmentation with Adversarial Data Augmentation: total=3 | keep=3 | review=0 | drop=0
[Image Filter] radiology / 2302.13057v2_Dee